In [0]:
# Define the Bronze storage location
bronze_base = "/Volumes/workspace/default/trucking_analytics/bronze"

# Define the Silver storage location
silver_base = "/Volumes/workspace/default/trucking_analytics/silver"

In [0]:
# Import required PySpark functions
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    when,
    current_timestamp
)

# Import Delta Lake functionality
from delta.tables import DeltaTable

In [0]:
# Read Customers from the Bronze Delta layer
customers_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/customers")
)

# Display sample records
display(customers_bronze.limit(30))

customer_id,customer_name,customer_type,credit_terms_days,primary_freight_type,account_status,contract_start_date,annual_revenue_potential,_ingested_at
CUST00001,Metro Wholesale,Dedicated,60,General,Inactive,2020-02-20,985117,2026-09-28T04:03:16.146Z
CUST00002,National Retail,Contract,30,Retail,Active,2021-06-02,4936566,2026-09-28T04:03:16.146Z
CUST00003,XYZ Industries,Contract,30,Consumer Goods,Active,2020-09-04,3102814,2026-09-28T04:03:16.146Z
CUST00004,American Corp,Contract,15,Food/Beverage,Active,2020-11-29,3948027,2026-09-28T04:03:16.146Z
CUST00005,American Distribution,Spot,45,Consumer Goods,Active,2020-02-04,3682564,2026-09-28T04:03:16.146Z
CUST00006,American Industries,Dedicated,30,Consumer Goods,Active,2021-02-16,973274,2026-09-28T04:03:16.146Z
CUST00007,Superior Distribution,Spot,30,Food/Beverage,Active,2021-02-05,1499297,2026-09-28T04:03:16.146Z
CUST00008,American Logistics,Contract,30,Automotive,Inactive,2021-07-23,1613275,2026-09-28T04:03:16.146Z
CUST00009,XYZ Wholesale,Contract,30,Retail,Inactive,2020-03-28,4713444,2026-09-28T04:03:16.146Z
CUST00010,National Logistics,Dedicated,45,Automotive,Active,2021-11-19,3468505,2026-09-28T04:03:16.146Z


In [0]:
# Clean and standardize the Customers data
customers_silver = (
    customers_bronze

    # Remove extra spaces from text columns
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("customer_type", trim(col("customer_type")))
    .withColumn("primary_freight_type", trim(col("primary_freight_type")))
    .withColumn("account_status", trim(col("account_status")))

    # Standardize categorical text values
    .withColumn("customer_type", upper(col("customer_type")))
    .withColumn("primary_freight_type", upper(col("primary_freight_type")))
    .withColumn("account_status", upper(col("account_status")))
)

# Display cleaned Customers data
display(customers_silver.limit(30))

customer_id,customer_name,customer_type,credit_terms_days,primary_freight_type,account_status,contract_start_date,annual_revenue_potential,_ingested_at
CUST00001,Metro Wholesale,DEDICATED,60,GENERAL,INACTIVE,2020-02-20,985117,2026-09-28T04:03:16.146Z
CUST00002,National Retail,CONTRACT,30,RETAIL,ACTIVE,2021-06-02,4936566,2026-09-28T04:03:16.146Z
CUST00003,XYZ Industries,CONTRACT,30,CONSUMER GOODS,ACTIVE,2020-09-04,3102814,2026-09-28T04:03:16.146Z
CUST00004,American Corp,CONTRACT,15,FOOD/BEVERAGE,ACTIVE,2020-11-29,3948027,2026-09-28T04:03:16.146Z
CUST00005,American Distribution,SPOT,45,CONSUMER GOODS,ACTIVE,2020-02-04,3682564,2026-09-28T04:03:16.146Z
CUST00006,American Industries,DEDICATED,30,CONSUMER GOODS,ACTIVE,2021-02-16,973274,2026-09-28T04:03:16.146Z
CUST00007,Superior Distribution,SPOT,30,FOOD/BEVERAGE,ACTIVE,2021-02-05,1499297,2026-09-28T04:03:16.146Z
CUST00008,American Logistics,CONTRACT,30,AUTOMOTIVE,INACTIVE,2021-07-23,1613275,2026-09-28T04:03:16.146Z
CUST00009,XYZ Wholesale,CONTRACT,30,RETAIL,INACTIVE,2020-03-28,4713444,2026-09-28T04:03:16.146Z
CUST00010,National Logistics,DEDICATED,45,AUTOMOTIVE,ACTIVE,2021-11-19,3468505,2026-09-28T04:03:16.146Z


In [0]:
# Check important fields for missing values
customers_silver.select(
    "customer_id",
    "customer_name",
    "customer_type",
    "account_status"
).where(
    col("customer_id").isNull() |
    col("customer_name").isNull()
).show(30, truncate=False)

+-----------+-------------+-------------+--------------+
|customer_id|customer_name|customer_type|account_status|
+-----------+-------------+-------------+--------------+
+-----------+-------------+-------------+--------------+



In [0]:
# Create a reusable function to write and merge data into Silver Delta tables

def write_to_silver(df, table_name, merge_condition):

    # Define the Silver table path
    silver_path = f"{silver_base}/{table_name}"

    # Create the Silver Delta table on the first run
    if not DeltaTable.isDeltaTable(spark, silver_path):

        df.write \
            .format("delta") \
            .save(silver_path)

    # Update existing records and insert new records
    else:

        silver_table = DeltaTable.forPath(
            spark,
            silver_path
        )

        (
            silver_table.alias("target")
            .merge(
                df.alias("source"),
                merge_condition
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    # Display the Silver table
    display(
        spark.read
        .format("delta")
        .load(silver_path)
        .limit(30)
    )

In [0]:
# Write Customers data to Silver using customer_id as the key
write_to_silver(
    customers_silver,
    "customers",
    "target.customer_id = source.customer_id"
)

customer_id,customer_name,customer_type,credit_terms_days,primary_freight_type,account_status,contract_start_date,annual_revenue_potential,_ingested_at
CUST00001,Metro Wholesale,DEDICATED,60,GENERAL,INACTIVE,2020-02-20,985117,2026-09-28T04:03:16.146Z
CUST00002,National Retail,CONTRACT,30,RETAIL,ACTIVE,2021-06-02,4936566,2026-09-28T04:03:16.146Z
CUST00003,XYZ Industries,CONTRACT,30,CONSUMER GOODS,ACTIVE,2020-09-04,3102814,2026-09-28T04:03:16.146Z
CUST00004,American Corp,CONTRACT,15,FOOD/BEVERAGE,ACTIVE,2020-11-29,3948027,2026-09-28T04:03:16.146Z
CUST00005,American Distribution,SPOT,45,CONSUMER GOODS,ACTIVE,2020-02-04,3682564,2026-09-28T04:03:16.146Z
CUST00006,American Industries,DEDICATED,30,CONSUMER GOODS,ACTIVE,2021-02-16,973274,2026-09-28T04:03:16.146Z
CUST00007,Superior Distribution,SPOT,30,FOOD/BEVERAGE,ACTIVE,2021-02-05,1499297,2026-09-28T04:03:16.146Z
CUST00008,American Logistics,CONTRACT,30,AUTOMOTIVE,INACTIVE,2021-07-23,1613275,2026-09-28T04:03:16.146Z
CUST00009,XYZ Wholesale,CONTRACT,30,RETAIL,INACTIVE,2020-03-28,4713444,2026-09-28T04:03:16.146Z
CUST00010,National Logistics,DEDICATED,45,AUTOMOTIVE,ACTIVE,2021-11-19,3468505,2026-09-28T04:03:16.146Z


In [0]:
# Read Drivers from the Bronze Delta layer
drivers_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/drivers")
)

# Display sample records
display(drivers_bronze.limit(30))

driver_id,first_name,last_name,hire_date,termination_date,license_number,license_state,date_of_birth,home_terminal,employment_status,cdl_class,years_experience,_ingested_at
DRV00001,Jennifer,Hernandez,2014-10-31,null,DL673510887,WA,1973-11-07,Denver,Active,A,3,2026-09-28T04:03:33.842Z
DRV00002,William,Martin,2020-10-02,null,DL128955006,GA,1976-11-03,Columbus,Active,A,20,2026-09-28T04:03:33.842Z
DRV00003,Charles,Hernandez,2021-09-21,null,DL523076025,NC,1970-04-06,Salt Lake City,Active,A,19,2026-09-28T04:03:33.842Z
DRV00004,Barbara,Brown,2013-09-08,null,DL735540030,WA,1995-02-06,Denver,Active,A,19,2026-09-28T04:03:33.842Z
DRV00005,Mary,Martinez,2018-12-02,null,DL706011277,AZ,1960-07-15,Chicago,Active,A,12,2026-09-28T04:03:33.842Z
DRV00006,Robert,Taylor,2019-04-04,null,DL811890569,NC,1990-05-02,Portland,Active,A,23,2026-09-28T04:03:33.842Z
DRV00007,James,Martinez,2019-07-02,2021-05-17,DL129100607,PA,1974-05-23,Miami,Terminated,A,5,2026-09-28T04:03:33.842Z
DRV00008,Barbara,Wilson,2020-06-09,2021-01-23,DL190137896,CO,1978-12-28,Philadelphia,Terminated,A,15,2026-09-28T04:03:33.842Z
DRV00009,Karen,Hernandez,2013-09-29,null,DL790147183,FL,1989-02-22,Los Angeles,Active,A,13,2026-09-28T04:03:33.842Z
DRV00010,Sarah,Gonzalez,2020-11-08,null,DL560219456,WA,1974-05-26,Minneapolis,Active,A,23,2026-09-28T04:03:33.842Z


In [0]:
# Clean and standardize the Drivers data
drivers_silver = (
    drivers_bronze

    # Remove extra spaces from text columns
    .withColumn("driver_id", trim(col("driver_id")))
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("license_number", trim(col("license_number")))
    .withColumn("license_state", trim(col("license_state")))
    .withColumn("home_terminal", trim(col("home_terminal")))
    .withColumn("employment_status", trim(col("employment_status")))
    .withColumn("cdl_class", trim(col("cdl_class")))

    # Standardize categorical values
    .withColumn("license_state", upper(col("license_state")))
    .withColumn("employment_status", upper(col("employment_status")))
    .withColumn("cdl_class", upper(col("cdl_class")))
)

# Display cleaned Drivers data
display(drivers_silver.limit(30))

driver_id,first_name,last_name,hire_date,termination_date,license_number,license_state,date_of_birth,home_terminal,employment_status,cdl_class,years_experience,_ingested_at
DRV00001,Jennifer,Hernandez,2014-10-31,null,DL673510887,WA,1973-11-07,Denver,ACTIVE,A,3,2026-09-28T04:03:33.842Z
DRV00002,William,Martin,2020-10-02,null,DL128955006,GA,1976-11-03,Columbus,ACTIVE,A,20,2026-09-28T04:03:33.842Z
DRV00003,Charles,Hernandez,2021-09-21,null,DL523076025,NC,1970-04-06,Salt Lake City,ACTIVE,A,19,2026-09-28T04:03:33.842Z
DRV00004,Barbara,Brown,2013-09-08,null,DL735540030,WA,1995-02-06,Denver,ACTIVE,A,19,2026-09-28T04:03:33.842Z
DRV00005,Mary,Martinez,2018-12-02,null,DL706011277,AZ,1960-07-15,Chicago,ACTIVE,A,12,2026-09-28T04:03:33.842Z
DRV00006,Robert,Taylor,2019-04-04,null,DL811890569,NC,1990-05-02,Portland,ACTIVE,A,23,2026-09-28T04:03:33.842Z
DRV00007,James,Martinez,2019-07-02,2021-05-17,DL129100607,PA,1974-05-23,Miami,TERMINATED,A,5,2026-09-28T04:03:33.842Z
DRV00008,Barbara,Wilson,2020-06-09,2021-01-23,DL190137896,CO,1978-12-28,Philadelphia,TERMINATED,A,15,2026-09-28T04:03:33.842Z
DRV00009,Karen,Hernandez,2013-09-29,null,DL790147183,FL,1989-02-22,Los Angeles,ACTIVE,A,13,2026-09-28T04:03:33.842Z
DRV00010,Sarah,Gonzalez,2020-11-08,null,DL560219456,WA,1974-05-26,Minneapolis,ACTIVE,A,23,2026-09-28T04:03:33.842Z


In [0]:
# Check important Driver fields for missing values
drivers_silver.select(
    "driver_id",
    "first_name",
    "last_name",
    "license_number"
).where(
    col("driver_id").isNull() |
    col("first_name").isNull() |
    col("last_name").isNull()
).show(30, truncate=False)

+---------+----------+---------+--------------+
|driver_id|first_name|last_name|license_number|
+---------+----------+---------+--------------+
+---------+----------+---------+--------------+



In [0]:
# Write Drivers data to Silver using driver_id as the key
write_to_silver(
    drivers_silver,
    "drivers",
    "target.driver_id = source.driver_id"
)

driver_id,first_name,last_name,hire_date,termination_date,license_number,license_state,date_of_birth,home_terminal,employment_status,cdl_class,years_experience,_ingested_at
DRV00001,Jennifer,Hernandez,2014-10-31,null,DL673510887,WA,1973-11-07,Denver,ACTIVE,A,3,2026-09-28T04:03:33.842Z
DRV00002,William,Martin,2020-10-02,null,DL128955006,GA,1976-11-03,Columbus,ACTIVE,A,20,2026-09-28T04:03:33.842Z
DRV00003,Charles,Hernandez,2021-09-21,null,DL523076025,NC,1970-04-06,Salt Lake City,ACTIVE,A,19,2026-09-28T04:03:33.842Z
DRV00004,Barbara,Brown,2013-09-08,null,DL735540030,WA,1995-02-06,Denver,ACTIVE,A,19,2026-09-28T04:03:33.842Z
DRV00005,Mary,Martinez,2018-12-02,null,DL706011277,AZ,1960-07-15,Chicago,ACTIVE,A,12,2026-09-28T04:03:33.842Z
DRV00006,Robert,Taylor,2019-04-04,null,DL811890569,NC,1990-05-02,Portland,ACTIVE,A,23,2026-09-28T04:03:33.842Z
DRV00007,James,Martinez,2019-07-02,2021-05-17,DL129100607,PA,1974-05-23,Miami,TERMINATED,A,5,2026-09-28T04:03:33.842Z
DRV00008,Barbara,Wilson,2020-06-09,2021-01-23,DL190137896,CO,1978-12-28,Philadelphia,TERMINATED,A,15,2026-09-28T04:03:33.842Z
DRV00009,Karen,Hernandez,2013-09-29,null,DL790147183,FL,1989-02-22,Los Angeles,ACTIVE,A,13,2026-09-28T04:03:33.842Z
DRV00010,Sarah,Gonzalez,2020-11-08,null,DL560219456,WA,1974-05-26,Minneapolis,ACTIVE,A,23,2026-09-28T04:03:33.842Z


In [0]:
# Read Trucks from the Bronze Delta layer
trucks_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/trucks")
)

# Display sample records
display(trucks_bronze.limit(30))

truck_id,unit_number,make,model_year,vin,acquisition_date,acquisition_mileage,fuel_type,tank_capacity_gallons,status,home_terminal,_ingested_at
TRK00001,3463,Peterbilt,2016,1VV205190335317039,2017-04-27,18814,Diesel,200,Active,Omaha,2026-09-28T04:03:42.883Z
TRK00002,6461,Kenworth,2015,1NV753749606229960,2018-10-04,26795,Diesel,150,Active,Seattle,2026-09-28T04:03:42.883Z
TRK00003,2335,Peterbilt,2018,1NT803101860493229,2020-09-17,11795,Diesel,150,Maintenance,Atlanta,2026-09-28T04:03:42.883Z
TRK00004,3317,Freightliner,2015,1VV131945381692351,2017-06-17,8875,Diesel,200,Active,Salt Lake City,2026-09-28T04:03:42.883Z
TRK00005,6731,Freightliner,2016,1FT822215709701985,2021-10-08,42042,Diesel,250,Maintenance,Miami,2026-09-28T04:03:42.883Z
TRK00006,6082,Kenworth,2017,1VV963516965177788,2018-05-12,47794,Diesel,150,Active,Minneapolis,2026-09-28T04:03:42.883Z
TRK00007,7421,Volvo,2019,1MV134100489809236,2020-01-29,42328,Diesel,200,Active,Charlotte,2026-09-28T04:03:42.883Z
TRK00008,3112,Mack,2015,1MV426140684859204,2015-01-27,27548,Diesel,150,Active,Houston,2026-09-28T04:03:42.883Z
TRK00009,2388,International,2015,1VU871233565012431,2016-11-04,24540,Diesel,250,Active,Denver,2026-09-28T04:03:42.883Z
TRK00010,6072,Volvo,2015,1MU182160377362421,2017-12-11,25066,Diesel,200,Active,Columbus,2026-09-28T04:03:42.883Z


In [0]:
# Clean and standardize the Trucks data
trucks_silver = (
    trucks_bronze

    # Remove extra spaces from text columns
    .withColumn("truck_id", trim(col("truck_id")))
    .withColumn("vin", trim(col("vin")))
    .withColumn("fuel_type", trim(col("fuel_type")))
    .withColumn("status", trim(col("status")))
    .withColumn("home_terminal", trim(col("home_terminal")))

    # Standardize categorical values
    .withColumn("fuel_type", upper(col("fuel_type")))
    .withColumn("status", upper(col("status")))
)

# Display cleaned Trucks data
display(trucks_silver.limit(30))

truck_id,unit_number,make,model_year,vin,acquisition_date,acquisition_mileage,fuel_type,tank_capacity_gallons,status,home_terminal,_ingested_at
TRK00001,3463,Peterbilt,2016,1VV205190335317039,2017-04-27,18814,DIESEL,200,ACTIVE,Omaha,2026-09-28T04:03:42.883Z
TRK00002,6461,Kenworth,2015,1NV753749606229960,2018-10-04,26795,DIESEL,150,ACTIVE,Seattle,2026-09-28T04:03:42.883Z
TRK00003,2335,Peterbilt,2018,1NT803101860493229,2020-09-17,11795,DIESEL,150,MAINTENANCE,Atlanta,2026-09-28T04:03:42.883Z
TRK00004,3317,Freightliner,2015,1VV131945381692351,2017-06-17,8875,DIESEL,200,ACTIVE,Salt Lake City,2026-09-28T04:03:42.883Z
TRK00005,6731,Freightliner,2016,1FT822215709701985,2021-10-08,42042,DIESEL,250,MAINTENANCE,Miami,2026-09-28T04:03:42.883Z
TRK00006,6082,Kenworth,2017,1VV963516965177788,2018-05-12,47794,DIESEL,150,ACTIVE,Minneapolis,2026-09-28T04:03:42.883Z
TRK00007,7421,Volvo,2019,1MV134100489809236,2020-01-29,42328,DIESEL,200,ACTIVE,Charlotte,2026-09-28T04:03:42.883Z
TRK00008,3112,Mack,2015,1MV426140684859204,2015-01-27,27548,DIESEL,150,ACTIVE,Houston,2026-09-28T04:03:42.883Z
TRK00009,2388,International,2015,1VU871233565012431,2016-11-04,24540,DIESEL,250,ACTIVE,Denver,2026-09-28T04:03:42.883Z
TRK00010,6072,Volvo,2015,1MU182160377362421,2017-12-11,25066,DIESEL,200,ACTIVE,Columbus,2026-09-28T04:03:42.883Z


In [0]:
# Check important Truck fields for missing values
trucks_silver.select(
    "truck_id",
    "unit_number",
    "vin",
    "status"
).where(
    col("truck_id").isNull() |
    col("vin").isNull()
).show(30, truncate=False)

+--------+-----------+---+------+
|truck_id|unit_number|vin|status|
+--------+-----------+---+------+
+--------+-----------+---+------+



In [0]:
# Write Trucks data to Silver using truck_id as the key
write_to_silver(
    trucks_silver,
    "trucks",
    "target.truck_id = source.truck_id"
)

truck_id,unit_number,make,model_year,vin,acquisition_date,acquisition_mileage,fuel_type,tank_capacity_gallons,status,home_terminal,_ingested_at
TRK00001,3463,Peterbilt,2016,1VV205190335317039,2017-04-27,18814,DIESEL,200,ACTIVE,Omaha,2026-09-28T04:03:42.883Z
TRK00002,6461,Kenworth,2015,1NV753749606229960,2018-10-04,26795,DIESEL,150,ACTIVE,Seattle,2026-09-28T04:03:42.883Z
TRK00003,2335,Peterbilt,2018,1NT803101860493229,2020-09-17,11795,DIESEL,150,MAINTENANCE,Atlanta,2026-09-28T04:03:42.883Z
TRK00004,3317,Freightliner,2015,1VV131945381692351,2017-06-17,8875,DIESEL,200,ACTIVE,Salt Lake City,2026-09-28T04:03:42.883Z
TRK00005,6731,Freightliner,2016,1FT822215709701985,2021-10-08,42042,DIESEL,250,MAINTENANCE,Miami,2026-09-28T04:03:42.883Z
TRK00006,6082,Kenworth,2017,1VV963516965177788,2018-05-12,47794,DIESEL,150,ACTIVE,Minneapolis,2026-09-28T04:03:42.883Z
TRK00007,7421,Volvo,2019,1MV134100489809236,2020-01-29,42328,DIESEL,200,ACTIVE,Charlotte,2026-09-28T04:03:42.883Z
TRK00008,3112,Mack,2015,1MV426140684859204,2015-01-27,27548,DIESEL,150,ACTIVE,Houston,2026-09-28T04:03:42.883Z
TRK00009,2388,International,2015,1VU871233565012431,2016-11-04,24540,DIESEL,250,ACTIVE,Denver,2026-09-28T04:03:42.883Z
TRK00010,6072,Volvo,2015,1MU182160377362421,2017-12-11,25066,DIESEL,200,ACTIVE,Columbus,2026-09-28T04:03:42.883Z


In [0]:
# Read Trailers from the Bronze Delta layer
trailers_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/trailers")
)

# Display sample records
display(trailers_bronze.limit(30))

trailer_id,trailer_number,trailer_type,length_feet,model_year,vin,acquisition_date,status,current_location,_ingested_at
TRL00001,4290,Refrigerated,53,2016,1AV889081755621178,2018-05-11,Active,Kansas City,2026-09-28T04:03:49.610Z
TRL00002,8848,Dry Van,53,2018,1BU942062588832828,2018-11-24,Active,Kansas City,2026-09-28T04:03:49.610Z
TRL00003,6147,Dry Van,53,2022,1AV669057807416326,2019-01-28,Active,Milwaukee,2026-09-28T04:03:49.610Z
TRL00004,3910,Dry Van,53,2024,1AV544000060905105,2018-08-27,Active,New York,2026-09-28T04:03:49.610Z
TRL00005,7524,Dry Van,53,2016,1AV170126369813275,2019-11-22,Active,Las Vegas,2026-09-28T04:03:49.610Z
TRL00006,6228,Refrigerated,53,2023,1BV247166900716104,2020-04-27,Active,Los Angeles,2026-09-28T04:03:49.610Z
TRL00007,5775,Dry Van,53,2015,1AU542428020643035,2016-03-15,Active,Indianapolis,2026-09-28T04:03:49.610Z
TRL00008,9044,Refrigerated,53,2020,1CV458098594511128,2019-04-23,Active,Oklahoma City,2026-09-28T04:03:49.610Z
TRL00009,4971,Dry Van,53,2020,1BU184087873073667,2021-05-14,Active,Kansas City,2026-09-28T04:03:49.610Z
TRL00010,6339,Refrigerated,53,2017,1AT967110002225980,2015-04-11,Active,Los Angeles,2026-09-28T04:03:49.610Z


In [0]:
# Clean and standardize the Trailers data
trailers_silver = (
    trailers_bronze

    # Remove extra spaces from text columns
    .withColumn("trailer_id", trim(col("trailer_id")))
    .withColumn("vin", trim(col("vin")))
    .withColumn("trailer_type", trim(col("trailer_type")))
    .withColumn("status", trim(col("status")))
    .withColumn("current_location", trim(col("current_location")))

    # Standardize categorical values
    .withColumn("trailer_type", upper(col("trailer_type")))
    .withColumn("status", upper(col("status")))
)

# Display cleaned Trailers data
display(trailers_silver.limit(30))

trailer_id,trailer_number,trailer_type,length_feet,model_year,vin,acquisition_date,status,current_location,_ingested_at
TRL00001,4290,REFRIGERATED,53,2016,1AV889081755621178,2018-05-11,ACTIVE,Kansas City,2026-09-28T04:03:49.610Z
TRL00002,8848,DRY VAN,53,2018,1BU942062588832828,2018-11-24,ACTIVE,Kansas City,2026-09-28T04:03:49.610Z
TRL00003,6147,DRY VAN,53,2022,1AV669057807416326,2019-01-28,ACTIVE,Milwaukee,2026-09-28T04:03:49.610Z
TRL00004,3910,DRY VAN,53,2024,1AV544000060905105,2018-08-27,ACTIVE,New York,2026-09-28T04:03:49.610Z
TRL00005,7524,DRY VAN,53,2016,1AV170126369813275,2019-11-22,ACTIVE,Las Vegas,2026-09-28T04:03:49.610Z
TRL00006,6228,REFRIGERATED,53,2023,1BV247166900716104,2020-04-27,ACTIVE,Los Angeles,2026-09-28T04:03:49.610Z
TRL00007,5775,DRY VAN,53,2015,1AU542428020643035,2016-03-15,ACTIVE,Indianapolis,2026-09-28T04:03:49.610Z
TRL00008,9044,REFRIGERATED,53,2020,1CV458098594511128,2019-04-23,ACTIVE,Oklahoma City,2026-09-28T04:03:49.610Z
TRL00009,4971,DRY VAN,53,2020,1BU184087873073667,2021-05-14,ACTIVE,Kansas City,2026-09-28T04:03:49.610Z
TRL00010,6339,REFRIGERATED,53,2017,1AT967110002225980,2015-04-11,ACTIVE,Los Angeles,2026-09-28T04:03:49.610Z


In [0]:
# Check important Trailer fields for missing values
trailers_silver.select(
    "trailer_id",
    "trailer_number",
    "trailer_type",
    "status"
).where(
    col("trailer_id").isNull() |
    col("trailer_number").isNull()
).show(30, truncate=False)

+----------+--------------+------------+------+
|trailer_id|trailer_number|trailer_type|status|
+----------+--------------+------------+------+
+----------+--------------+------------+------+



In [0]:
# Write Trailers data to Silver using trailer_id as the key
write_to_silver(
    trailers_silver,
    "trailers",
    "target.trailer_id = source.trailer_id"
)

trailer_id,trailer_number,trailer_type,length_feet,model_year,vin,acquisition_date,status,current_location,_ingested_at
TRL00001,4290,REFRIGERATED,53,2016,1AV889081755621178,2018-05-11,ACTIVE,Kansas City,2026-09-28T04:03:49.610Z
TRL00002,8848,DRY VAN,53,2018,1BU942062588832828,2018-11-24,ACTIVE,Kansas City,2026-09-28T04:03:49.610Z
TRL00003,6147,DRY VAN,53,2022,1AV669057807416326,2019-01-28,ACTIVE,Milwaukee,2026-09-28T04:03:49.610Z
TRL00004,3910,DRY VAN,53,2024,1AV544000060905105,2018-08-27,ACTIVE,New York,2026-09-28T04:03:49.610Z
TRL00005,7524,DRY VAN,53,2016,1AV170126369813275,2019-11-22,ACTIVE,Las Vegas,2026-09-28T04:03:49.610Z
TRL00006,6228,REFRIGERATED,53,2023,1BV247166900716104,2020-04-27,ACTIVE,Los Angeles,2026-09-28T04:03:49.610Z
TRL00007,5775,DRY VAN,53,2015,1AU542428020643035,2016-03-15,ACTIVE,Indianapolis,2026-09-28T04:03:49.610Z
TRL00008,9044,REFRIGERATED,53,2020,1CV458098594511128,2019-04-23,ACTIVE,Oklahoma City,2026-09-28T04:03:49.610Z
TRL00009,4971,DRY VAN,53,2020,1BU184087873073667,2021-05-14,ACTIVE,Kansas City,2026-09-28T04:03:49.610Z
TRL00010,6339,REFRIGERATED,53,2017,1AT967110002225980,2015-04-11,ACTIVE,Los Angeles,2026-09-28T04:03:49.610Z


In [0]:
# Read Facilities from the Bronze Delta layer
facilities_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/facilities")
)

# Display sample records
display(facilities_bronze.limit(30))

facility_id,facility_name,facility_type,city,state,latitude,longitude,dock_doors,operating_hours,_ingested_at
FAC00001,Houston Distribution Center,Cross-Dock,Houston,TX,29.7604,-95.3698,125,24/7,2026-09-28T04:03:58.268Z
FAC00002,Kansas City Hub,Cross-Dock,Kansas City,MO,39.0997,-94.5786,33,7AM-7PM,2026-09-28T04:03:58.268Z
FAC00003,Charlotte Distribution Center,Distribution Center,Charlotte,NC,35.2271,-80.8431,138,24/7,2026-09-28T04:03:58.268Z
FAC00004,Chicago Distribution Center,Distribution Center,Chicago,IL,41.8781,-87.6298,38,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00005,Indianapolis Hub,Distribution Center,Indianapolis,IN,39.7684,-86.1581,56,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00006,Atlanta Warehouse,Distribution Center,Atlanta,GA,33.749,-84.388,149,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00007,Detroit Warehouse,Terminal,Detroit,MI,42.3314,-83.0458,44,24/7,2026-09-28T04:03:58.268Z
FAC00008,Philadelphia Hub,Cross-Dock,Philadelphia,PA,39.9526,-75.1652,110,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00009,Nashville Distribution Center,Terminal,Nashville,TN,36.1627,-86.7816,115,6AM-10PM,2026-09-28T04:03:58.268Z
FAC00010,Indianapolis Distribution Center,Warehouse,Indianapolis,IN,39.7684,-86.1581,92,6AM-10PM,2026-09-28T04:03:58.268Z


In [0]:
# Clean and standardize the Facilities data
facilities_silver = (
    facilities_bronze

    # Remove extra spaces from text columns
    .withColumn("facility_id", trim(col("facility_id")))
    .withColumn("facility_name", trim(col("facility_name")))
    .withColumn("facility_type", trim(col("facility_type")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("operating_hours", trim(col("operating_hours")))

    # Standardize categorical values
    .withColumn("facility_type", upper(col("facility_type")))
    .withColumn("state", upper(col("state")))
)

# Display cleaned Facilities data
display(facilities_silver.limit(30))

facility_id,facility_name,facility_type,city,state,latitude,longitude,dock_doors,operating_hours,_ingested_at
FAC00001,Houston Distribution Center,CROSS-DOCK,Houston,TX,29.7604,-95.3698,125,24/7,2026-09-28T04:03:58.268Z
FAC00002,Kansas City Hub,CROSS-DOCK,Kansas City,MO,39.0997,-94.5786,33,7AM-7PM,2026-09-28T04:03:58.268Z
FAC00003,Charlotte Distribution Center,DISTRIBUTION CENTER,Charlotte,NC,35.2271,-80.8431,138,24/7,2026-09-28T04:03:58.268Z
FAC00004,Chicago Distribution Center,DISTRIBUTION CENTER,Chicago,IL,41.8781,-87.6298,38,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00005,Indianapolis Hub,DISTRIBUTION CENTER,Indianapolis,IN,39.7684,-86.1581,56,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00006,Atlanta Warehouse,DISTRIBUTION CENTER,Atlanta,GA,33.749,-84.388,149,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00007,Detroit Warehouse,TERMINAL,Detroit,MI,42.3314,-83.0458,44,24/7,2026-09-28T04:03:58.268Z
FAC00008,Philadelphia Hub,CROSS-DOCK,Philadelphia,PA,39.9526,-75.1652,110,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00009,Nashville Distribution Center,TERMINAL,Nashville,TN,36.1627,-86.7816,115,6AM-10PM,2026-09-28T04:03:58.268Z
FAC00010,Indianapolis Distribution Center,WAREHOUSE,Indianapolis,IN,39.7684,-86.1581,92,6AM-10PM,2026-09-28T04:03:58.268Z


In [0]:
# Check important Facility fields for missing values
facilities_silver.select(
    "facility_id",
    "facility_name",
    "city",
    "state"
).where(
    col("facility_id").isNull() |
    col("facility_name").isNull()
).show(30, truncate=False)

+-----------+-------------+----+-----+
|facility_id|facility_name|city|state|
+-----------+-------------+----+-----+
+-----------+-------------+----+-----+



In [0]:
# Write Facilities data to Silver using facility_id as the key
write_to_silver(
    facilities_silver,
    "facilities",
    "target.facility_id = source.facility_id"
)

facility_id,facility_name,facility_type,city,state,latitude,longitude,dock_doors,operating_hours,_ingested_at
FAC00001,Houston Distribution Center,CROSS-DOCK,Houston,TX,29.7604,-95.3698,125,24/7,2026-09-28T04:03:58.268Z
FAC00002,Kansas City Hub,CROSS-DOCK,Kansas City,MO,39.0997,-94.5786,33,7AM-7PM,2026-09-28T04:03:58.268Z
FAC00003,Charlotte Distribution Center,DISTRIBUTION CENTER,Charlotte,NC,35.2271,-80.8431,138,24/7,2026-09-28T04:03:58.268Z
FAC00004,Chicago Distribution Center,DISTRIBUTION CENTER,Chicago,IL,41.8781,-87.6298,38,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00005,Indianapolis Hub,DISTRIBUTION CENTER,Indianapolis,IN,39.7684,-86.1581,56,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00006,Atlanta Warehouse,DISTRIBUTION CENTER,Atlanta,GA,33.749,-84.388,149,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00007,Detroit Warehouse,TERMINAL,Detroit,MI,42.3314,-83.0458,44,24/7,2026-09-28T04:03:58.268Z
FAC00008,Philadelphia Hub,CROSS-DOCK,Philadelphia,PA,39.9526,-75.1652,110,8AM-5PM,2026-09-28T04:03:58.268Z
FAC00009,Nashville Distribution Center,TERMINAL,Nashville,TN,36.1627,-86.7816,115,6AM-10PM,2026-09-28T04:03:58.268Z
FAC00010,Indianapolis Distribution Center,WAREHOUSE,Indianapolis,IN,39.7684,-86.1581,92,6AM-10PM,2026-09-28T04:03:58.268Z


In [0]:
# Read Routes from the Bronze Delta layer
routes_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/routes")
)

# Display sample records
display(routes_bronze.limit(30))

route_id,origin_city,origin_state,destination_city,destination_state,typical_distance_miles,base_rate_per_mile,fuel_surcharge_rate,typical_transit_days,_ingested_at
RTE00001,Atlanta,GA,Chicago,IL,677,1.7,0.19,1,2026-09-28T04:04:05.599Z
RTE00002,Atlanta,GA,Miami,FL,697,2.08,0.22,1,2026-09-28T04:04:05.599Z
RTE00003,Chicago,IL,Los Angeles,CA,2003,2.55,0.29,3,2026-09-28T04:04:05.599Z
RTE00004,Chicago,IL,Portland,OR,2017,1.84,0.19,3,2026-09-28T04:04:05.599Z
RTE00005,Chicago,IL,Las Vegas,NV,1749,2.41,0.26,3,2026-09-28T04:04:05.599Z
RTE00006,Chicago,IL,Charlotte,NC,675,2.13,0.28,1,2026-09-28T04:04:05.599Z
RTE00007,Dallas,TX,New York,NY,1576,2.54,0.32,2,2026-09-28T04:04:05.599Z
RTE00008,Dallas,TX,Denver,CO,762,2.62,0.15,1,2026-09-28T04:04:05.599Z
RTE00009,Dallas,TX,Indianapolis,IN,878,1.95,0.17,1,2026-09-28T04:04:05.599Z
RTE00010,New York,NY,Philadelphia,PA,92,1.61,0.17,1,2026-09-28T04:04:05.599Z


In [0]:
# Clean and standardize the Routes data
routes_silver = (
    routes_bronze

    # Remove extra spaces from text columns
    .withColumn("route_id", trim(col("route_id")))
    .withColumn("origin_city", trim(col("origin_city")))
    .withColumn("origin_state", trim(col("origin_state")))
    .withColumn("destination_city", trim(col("destination_city")))
    .withColumn("destination_state", trim(col("destination_state")))

    # Standardize state values
    .withColumn("origin_state", upper(col("origin_state")))
    .withColumn("destination_state", upper(col("destination_state")))
)

# Display cleaned Routes data
display(routes_silver.limit(30))

route_id,origin_city,origin_state,destination_city,destination_state,typical_distance_miles,base_rate_per_mile,fuel_surcharge_rate,typical_transit_days,_ingested_at
RTE00001,Atlanta,GA,Chicago,IL,677,1.7,0.19,1,2026-09-28T04:04:05.599Z
RTE00002,Atlanta,GA,Miami,FL,697,2.08,0.22,1,2026-09-28T04:04:05.599Z
RTE00003,Chicago,IL,Los Angeles,CA,2003,2.55,0.29,3,2026-09-28T04:04:05.599Z
RTE00004,Chicago,IL,Portland,OR,2017,1.84,0.19,3,2026-09-28T04:04:05.599Z
RTE00005,Chicago,IL,Las Vegas,NV,1749,2.41,0.26,3,2026-09-28T04:04:05.599Z
RTE00006,Chicago,IL,Charlotte,NC,675,2.13,0.28,1,2026-09-28T04:04:05.599Z
RTE00007,Dallas,TX,New York,NY,1576,2.54,0.32,2,2026-09-28T04:04:05.599Z
RTE00008,Dallas,TX,Denver,CO,762,2.62,0.15,1,2026-09-28T04:04:05.599Z
RTE00009,Dallas,TX,Indianapolis,IN,878,1.95,0.17,1,2026-09-28T04:04:05.599Z
RTE00010,New York,NY,Philadelphia,PA,92,1.61,0.17,1,2026-09-28T04:04:05.599Z


In [0]:
# Check important Route fields for missing values
routes_silver.select(
    "route_id",
    "origin_city",
    "origin_state",
    "destination_city",
    "destination_state"
).where(
    col("route_id").isNull() |
    col("origin_city").isNull() |
    col("destination_city").isNull()
).show(30, truncate=False)

+--------+-----------+------------+----------------+-----------------+
|route_id|origin_city|origin_state|destination_city|destination_state|
+--------+-----------+------------+----------------+-----------------+
+--------+-----------+------------+----------------+-----------------+



In [0]:
# Write Routes data to Silver using route_id as the key
write_to_silver(
    routes_silver,
    "routes",
    "target.route_id = source.route_id"
)

route_id,origin_city,origin_state,destination_city,destination_state,typical_distance_miles,base_rate_per_mile,fuel_surcharge_rate,typical_transit_days,_ingested_at
RTE00001,Atlanta,GA,Chicago,IL,677,1.7,0.19,1,2026-09-28T04:04:05.599Z
RTE00002,Atlanta,GA,Miami,FL,697,2.08,0.22,1,2026-09-28T04:04:05.599Z
RTE00003,Chicago,IL,Los Angeles,CA,2003,2.55,0.29,3,2026-09-28T04:04:05.599Z
RTE00004,Chicago,IL,Portland,OR,2017,1.84,0.19,3,2026-09-28T04:04:05.599Z
RTE00005,Chicago,IL,Las Vegas,NV,1749,2.41,0.26,3,2026-09-28T04:04:05.599Z
RTE00006,Chicago,IL,Charlotte,NC,675,2.13,0.28,1,2026-09-28T04:04:05.599Z
RTE00007,Dallas,TX,New York,NY,1576,2.54,0.32,2,2026-09-28T04:04:05.599Z
RTE00008,Dallas,TX,Denver,CO,762,2.62,0.15,1,2026-09-28T04:04:05.599Z
RTE00009,Dallas,TX,Indianapolis,IN,878,1.95,0.17,1,2026-09-28T04:04:05.599Z
RTE00010,New York,NY,Philadelphia,PA,92,1.61,0.17,1,2026-09-28T04:04:05.599Z


In [0]:
# Read Loads from the Bronze Delta layer
loads_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/loads")
)

# Display sample records
display(loads_bronze.limit(30))

load_id,customer_id,route_id,load_date,load_type,weight_lbs,pieces,revenue,fuel_surcharge,accessorial_charges,load_status,booking_type,_ingested_at
LOAD00060059,CUST00085,RTE00056,2024-02-12,Refrigerated,35771,2,4494.57,309.24,0,Completed,Dedicated,2026-09-28T04:04:13.260Z
LOAD00060060,CUST00054,RTE00014,2024-02-12,Dry Van,23132,8,2193.62,228.76,100,Completed,Spot,2026-09-28T04:04:13.260Z
LOAD00060061,CUST00017,RTE00036,2024-02-12,Refrigerated,17307,5,698.0,86.46,150,Completed,Spot,2026-09-28T04:04:13.260Z
LOAD00060062,CUST00178,RTE00048,2024-02-12,Dry Van,27889,5,5853.15,792.88,150,Completed,Dedicated,2026-09-28T04:04:13.260Z
LOAD00060063,CUST00089,RTE00006,2024-02-12,Dry Van,28561,4,1577.02,189.0,75,Completed,Contract,2026-09-28T04:04:13.260Z
LOAD00060064,CUST00031,RTE00057,2024-02-12,Refrigerated,40020,22,2196.49,276.3,75,Completed,Dedicated,2026-09-28T04:04:13.260Z
LOAD00060065,CUST00069,RTE00046,2024-02-12,Refrigerated,43255,8,5815.07,499.18,200,Completed,Dedicated,2026-09-28T04:04:13.260Z
LOAD00060066,CUST00200,RTE00023,2024-02-12,Dry Van,22524,7,3191.96,216.92,0,Completed,Dedicated,2026-09-28T04:04:13.260Z
LOAD00060067,CUST00113,RTE00011,2024-02-12,Dry Van,25284,27,852.88,131.28,75,Completed,Dedicated,2026-09-28T04:04:13.260Z
LOAD00060068,CUST00199,RTE00011,2024-02-12,Dry Van,22736,7,962.55,131.28,200,Completed,Dedicated,2026-09-28T04:04:13.260Z


In [0]:
# Clean and standardize the Loads data
loads_silver = (
    loads_bronze

    # Remove extra spaces from text columns
    .withColumn("load_id", trim(col("load_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("route_id", trim(col("route_id")))
    .withColumn("load_type", trim(col("load_type")))
    .withColumn("load_status", trim(col("load_status")))
    .withColumn("booking_type", trim(col("booking_type")))

    # Standardize categorical values
    .withColumn("load_type", upper(col("load_type")))
    .withColumn("load_status", upper(col("load_status")))
    .withColumn("booking_type", upper(col("booking_type")))
)

# Display cleaned Loads data
display(loads_silver.limit(30))

load_id,customer_id,route_id,load_date,load_type,weight_lbs,pieces,revenue,fuel_surcharge,accessorial_charges,load_status,booking_type,_ingested_at
LOAD00060059,CUST00085,RTE00056,2024-02-12,REFRIGERATED,35771,2,4494.57,309.24,0,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00060060,CUST00054,RTE00014,2024-02-12,DRY VAN,23132,8,2193.62,228.76,100,COMPLETED,SPOT,2026-09-28T04:04:13.260Z
LOAD00060061,CUST00017,RTE00036,2024-02-12,REFRIGERATED,17307,5,698.0,86.46,150,COMPLETED,SPOT,2026-09-28T04:04:13.260Z
LOAD00060062,CUST00178,RTE00048,2024-02-12,DRY VAN,27889,5,5853.15,792.88,150,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00060063,CUST00089,RTE00006,2024-02-12,DRY VAN,28561,4,1577.02,189.0,75,COMPLETED,CONTRACT,2026-09-28T04:04:13.260Z
LOAD00060064,CUST00031,RTE00057,2024-02-12,REFRIGERATED,40020,22,2196.49,276.3,75,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00060065,CUST00069,RTE00046,2024-02-12,REFRIGERATED,43255,8,5815.07,499.18,200,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00060066,CUST00200,RTE00023,2024-02-12,DRY VAN,22524,7,3191.96,216.92,0,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00060067,CUST00113,RTE00011,2024-02-12,DRY VAN,25284,27,852.88,131.28,75,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00060068,CUST00199,RTE00011,2024-02-12,DRY VAN,22736,7,962.55,131.28,200,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z


In [0]:
# Check important Load fields for missing values
loads_silver.select(
    "load_id",
    "customer_id",
    "route_id",
    "load_date",
    "revenue"
).where(
    col("load_id").isNull() |
    col("customer_id").isNull() |
    col("route_id").isNull()
).show(30, truncate=False)

+-------+-----------+--------+---------+-------+
|load_id|customer_id|route_id|load_date|revenue|
+-------+-----------+--------+---------+-------+
+-------+-----------+--------+---------+-------+



In [0]:
# Write Loads data to Silver using load_id as the key
write_to_silver(
    loads_silver,
    "loads",
    "target.load_id = source.load_id"
)

load_id,customer_id,route_id,load_date,load_type,weight_lbs,pieces,revenue,fuel_surcharge,accessorial_charges,load_status,booking_type,_ingested_at
LOAD00036038,CUST00030,RTE00007,2023-04-06,REFRIGERATED,29343,4,4425.24,504.32,0,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00036039,CUST00150,RTE00021,2023-04-06,DRY VAN,26651,11,5901.89,590.24,0,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00036040,CUST00043,RTE00022,2023-04-06,REFRIGERATED,42082,18,1128.6,161.53,0,COMPLETED,SPOT,2026-09-28T04:04:13.260Z
LOAD00036041,CUST00177,RTE00010,2023-04-06,DRY VAN,14826,1,143.61,15.64,0,COMPLETED,SPOT,2026-09-28T04:04:13.260Z
LOAD00036042,CUST00156,RTE00032,2023-04-06,DRY VAN,27493,25,4570.13,451.8,75,COMPLETED,CONTRACT,2026-09-28T04:04:13.260Z
LOAD00036043,CUST00117,RTE00056,2023-04-06,DRY VAN,19839,2,4157.24,309.24,150,COMPLETED,DEDICATED,2026-09-28T04:04:13.260Z
LOAD00036044,CUST00113,RTE00050,2023-04-06,DRY VAN,15351,20,1441.41,170.88,0,COMPLETED,CONTRACT,2026-09-28T04:04:13.260Z
LOAD00036045,CUST00023,RTE00056,2023-04-06,DRY VAN,17958,22,4374.37,309.24,75,COMPLETED,SPOT,2026-09-28T04:04:13.260Z
LOAD00036046,CUST00101,RTE00001,2023-04-06,REFRIGERATED,27504,11,1258.84,128.63,100,COMPLETED,SPOT,2026-09-28T04:04:13.260Z
LOAD00036047,CUST00196,RTE00006,2023-04-06,DRY VAN,19113,1,1373.96,189.0,75,COMPLETED,SPOT,2026-09-28T04:04:13.260Z


In [0]:
# Read Trips from the Bronze Delta layer
trips_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/trips")
)

# Display sample records
display(trips_bronze.limit(30))

trip_id,load_id,driver_id,truck_id,trailer_id,dispatch_date,actual_distance_miles,actual_duration_hours,fuel_gallons_used,average_mpg,idle_time_hours,trip_status,_ingested_at
TRIP00012015,LOAD00012015,DRV00133,TRK00078,TRL00109,2022-06-03,3204,50.2,566.1,5.66,11.1,Completed,2026-09-28T04:04:22.520Z
TRIP00012016,LOAD00012016,DRV00111,TRK00078,TRL00069,2022-06-03,90,1.5,15.8,5.7,3.5,Completed,2026-09-28T04:04:22.520Z
TRIP00012017,LOAD00012017,DRV00123,TRK00038,TRL00094,2022-06-03,647,10.3,102.1,6.34,11.3,Completed,2026-09-28T04:04:22.520Z
TRIP00012018,LOAD00012018,DRV00036,TRK00103,TRL00149,2022-06-03,281,4.6,43.6,6.44,3.0,Completed,2026-09-28T04:04:22.520Z
TRIP00012019,LOAD00012019,DRV00001,TRK00018,TRL00102,2022-06-03,272,4.5,44.7,6.09,2.0,Completed,2026-09-28T04:04:22.520Z
TRIP00012020,LOAD00012020,DRV00020,TRK00107,TRL00075,2022-06-03,2727,42.8,408.2,6.68,8.1,Completed,2026-09-28T04:04:22.520Z
TRIP00012021,LOAD00012021,DRV00078,TRK00054,TRL00135,2022-06-04,1259,21.6,212.0,5.94,4.9,Completed,2026-09-28T04:04:22.520Z
TRIP00012022,LOAD00012022,DRV00071,TRK00060,TRL00123,2022-06-04,1225,22.9,216.8,5.65,8.4,Completed,2026-09-28T04:04:22.520Z
TRIP00012023,LOAD00012023,DRV00063,TRK00103,TRL00083,2022-06-04,707,11.1,95.9,7.37,2.6,Completed,2026-09-28T04:04:22.520Z
TRIP00012024,LOAD00012024,DRV00062,TRK00051,TRL00047,2022-06-04,2212,39.0,368.1,6.01,3.0,Completed,2026-09-28T04:04:22.520Z


In [0]:
# Clean and standardize the Trips data
trips_silver = (
    trips_bronze

    # Remove extra spaces from ID columns
    .withColumn("trip_id", trim(col("trip_id")))
    .withColumn("load_id", trim(col("load_id")))
    .withColumn("driver_id", trim(col("driver_id")))
    .withColumn("truck_id", trim(col("truck_id")))
    .withColumn("trailer_id", trim(col("trailer_id")))
    .withColumn("trip_status", trim(col("trip_status")))

    # Standardize trip status
    .withColumn("trip_status", upper(col("trip_status")))
)

# Display cleaned Trips data
display(trips_silver.limit(30))

trip_id,load_id,driver_id,truck_id,trailer_id,dispatch_date,actual_distance_miles,actual_duration_hours,fuel_gallons_used,average_mpg,idle_time_hours,trip_status,_ingested_at
TRIP00012015,LOAD00012015,DRV00133,TRK00078,TRL00109,2022-06-03,3204,50.2,566.1,5.66,11.1,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012016,LOAD00012016,DRV00111,TRK00078,TRL00069,2022-06-03,90,1.5,15.8,5.7,3.5,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012017,LOAD00012017,DRV00123,TRK00038,TRL00094,2022-06-03,647,10.3,102.1,6.34,11.3,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012018,LOAD00012018,DRV00036,TRK00103,TRL00149,2022-06-03,281,4.6,43.6,6.44,3.0,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012019,LOAD00012019,DRV00001,TRK00018,TRL00102,2022-06-03,272,4.5,44.7,6.09,2.0,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012020,LOAD00012020,DRV00020,TRK00107,TRL00075,2022-06-03,2727,42.8,408.2,6.68,8.1,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012021,LOAD00012021,DRV00078,TRK00054,TRL00135,2022-06-04,1259,21.6,212.0,5.94,4.9,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012022,LOAD00012022,DRV00071,TRK00060,TRL00123,2022-06-04,1225,22.9,216.8,5.65,8.4,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012023,LOAD00012023,DRV00063,TRK00103,TRL00083,2022-06-04,707,11.1,95.9,7.37,2.6,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00012024,LOAD00012024,DRV00062,TRK00051,TRL00047,2022-06-04,2212,39.0,368.1,6.01,3.0,COMPLETED,2026-09-28T04:04:22.520Z


In [0]:
# Check important Trip fields for missing values
trips_silver.select(
    "trip_id",
    "load_id",
    "driver_id",
    "truck_id",
    "dispatch_date"
).where(
    col("trip_id").isNull() |
    col("load_id").isNull() |
    col("driver_id").isNull() |
    col("truck_id").isNull()
).show(30, truncate=False)

+------------+------------+---------+--------+-------------+
|trip_id     |load_id     |driver_id|truck_id|dispatch_date|
+------------+------------+---------+--------+-------------+
|TRIP00036064|LOAD00036064|DRV00110 |NULL    |2023-04-06   |
|TRIP00036126|LOAD00036126|DRV00013 |NULL    |2023-04-07   |
|TRIP00036166|LOAD00036166|NULL     |TRK00050|2023-04-08   |
|TRIP00036191|LOAD00036191|NULL     |TRK00071|2023-04-08   |
|TRIP00036206|LOAD00036206|DRV00146 |NULL    |2023-04-08   |
|TRIP00036221|LOAD00036221|DRV00084 |NULL    |2023-04-09   |
|TRIP00036260|LOAD00036260|NULL     |TRK00035|2023-04-09   |
|TRIP00036310|LOAD00036310|NULL     |TRK00002|2023-04-10   |
|TRIP00036336|LOAD00036336|NULL     |TRK00008|2023-04-10   |
|TRIP00036338|LOAD00036338|NULL     |TRK00006|2023-04-10   |
|TRIP00036353|LOAD00036353|DRV00028 |NULL    |2023-04-10   |
|TRIP00036370|LOAD00036370|NULL     |TRK00025|2023-04-10   |
|TRIP00036393|LOAD00036393|DRV00109 |NULL    |2023-04-11   |
|TRIP00036396|LOAD000363

In [0]:
# Write Trips data to Silver using trip_id as the key
write_to_silver(
    trips_silver,
    "trips",
    "target.trip_id = source.trip_id"
)

trip_id,load_id,driver_id,truck_id,trailer_id,dispatch_date,actual_distance_miles,actual_duration_hours,fuel_gallons_used,average_mpg,idle_time_hours,trip_status,_ingested_at
TRIP00036040,LOAD00036040,DRV00069,TRK00084,TRL00036,2023-04-06,574,10.8,78.2,7.34,5.0,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036041,LOAD00036041,DRV00080,TRK00071,TRL00099,2023-04-06,99,1.6,17.6,5.64,6.9,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036042,LOAD00036042,DRV00058,TRK00114,TRL00024,2023-04-06,2387,42.2,366.7,6.51,2.8,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036043,LOAD00036043,DRV00117,TRK00016,TRL00079,2023-04-06,1694,32.5,249.1,6.8,2.6,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036044,LOAD00036044,DRV00030,TRK00040,TRL00149,2023-04-06,731,12.2,113.0,6.47,10.7,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036045,LOAD00036045,DRV00016,TRK00065,TRL00037,2023-04-06,1853,31.0,319.5,5.8,2.9,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036046,LOAD00036046,DRV00132,TRK00098,TRL00114,2023-04-06,673,12.3,118.7,5.67,10.2,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036047,LOAD00036047,DRV00147,TRK00002,TRL00081,2023-04-06,686,11.0,103.5,6.63,5.8,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036048,LOAD00036048,DRV00060,TRK00044,TRL00071,2023-04-06,650,10.6,89.7,7.25,6.7,COMPLETED,2026-09-28T04:04:22.520Z
TRIP00036049,LOAD00036049,DRV00001,TRK00054,TRL00095,2023-04-06,689,11.5,116.0,5.94,7.3,COMPLETED,2026-09-28T04:04:22.520Z


In [0]:
# Read Fuel Purchases from the Bronze Delta layer
fuel_purchases_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/fuel_purchases")
)

# Display sample records
display(fuel_purchases_bronze.limit(30))

fuel_purchase_id,trip_id,truck_id,driver_id,purchase_date,location_city,location_state,gallons,price_per_gallon,total_cost,fuel_card_number,_ingested_at
FUEL00180736,TRIP00016151,TRK00001,DRV00045,2022-07-27T16:00:00.000Z,Omaha,TX,77.0,4.331,333.49,FC569510,2026-09-28T04:04:31.137Z
FUEL00180737,TRIP00074093,TRK00015,DRV00002,2024-08-09T21:00:00.000Z,Portland,GA,63.0,3.881,244.5,FC808591,2026-09-28T04:04:31.137Z
FUEL00180738,TRIP00018959,TRK00112,DRV00041,2022-08-30T02:00:00.000Z,Philadelphia,NC,174.8,4.099,716.51,FC200601,2026-09-28T04:04:31.137Z
FUEL00180739,TRIP00020647,TRK00067,DRV00016,2022-09-23T16:00:00.000Z,Phoenix,CO,195.1,4.242,827.61,FC736730,2026-09-28T04:04:31.137Z
FUEL00180740,TRIP00068556,TRK00070,DRV00133,2024-06-01T01:00:00.000Z,New York,OH,85.4,3.269,279.17,FC612291,2026-09-28T04:04:31.137Z
FUEL00180741,TRIP00003001,TRK00092,null,2022-02-08T00:00:00.000Z,Portland,MO,83.6,3.937,329.13,FC140257,2026-09-28T04:04:31.137Z
FUEL00180742,TRIP00064359,TRK00106,DRV00010,2024-04-08T12:00:00.000Z,Houston,MI,60.7,3.413,207.17,FC864332,2026-09-28T04:04:31.137Z
FUEL00180743,TRIP00044980,TRK00091,DRV00019,2023-08-02T18:00:00.000Z,Los Angeles,OH,147.1,4.007,589.43,FC309209,2026-09-28T04:04:31.137Z
FUEL00180744,TRIP00063420,TRK00095,DRV00110,2024-03-27T17:00:00.000Z,Seattle,IN,63.9,3.985,254.64,FC938982,2026-09-28T04:04:31.137Z
FUEL00180745,TRIP00050301,TRK00113,DRV00005,2023-10-11T03:00:00.000Z,New York,FL,176.6,3.798,670.73,FC941614,2026-09-28T04:04:31.137Z


In [0]:
# Clean and standardize the Fuel Purchases data
fuel_purchases_silver = (
    fuel_purchases_bronze

    # Remove extra spaces from ID and location columns
    .withColumn("fuel_purchase_id", trim(col("fuel_purchase_id")))
    .withColumn("trip_id", trim(col("trip_id")))
    .withColumn("truck_id", trim(col("truck_id")))
    .withColumn("driver_id", trim(col("driver_id")))
    .withColumn("location_city", trim(col("location_city")))
    .withColumn("location_state", trim(col("location_state")))
    .withColumn("fuel_card_number", trim(col("fuel_card_number")))

    # Standardize state values
    .withColumn("location_state", upper(col("location_state")))
)

# Display cleaned Fuel Purchases data
display(fuel_purchases_silver.limit(30))

fuel_purchase_id,trip_id,truck_id,driver_id,purchase_date,location_city,location_state,gallons,price_per_gallon,total_cost,fuel_card_number,_ingested_at
FUEL00180736,TRIP00016151,TRK00001,DRV00045,2022-07-27T16:00:00.000Z,Omaha,TX,77.0,4.331,333.49,FC569510,2026-09-28T04:04:31.137Z
FUEL00180737,TRIP00074093,TRK00015,DRV00002,2024-08-09T21:00:00.000Z,Portland,GA,63.0,3.881,244.5,FC808591,2026-09-28T04:04:31.137Z
FUEL00180738,TRIP00018959,TRK00112,DRV00041,2022-08-30T02:00:00.000Z,Philadelphia,NC,174.8,4.099,716.51,FC200601,2026-09-28T04:04:31.137Z
FUEL00180739,TRIP00020647,TRK00067,DRV00016,2022-09-23T16:00:00.000Z,Phoenix,CO,195.1,4.242,827.61,FC736730,2026-09-28T04:04:31.137Z
FUEL00180740,TRIP00068556,TRK00070,DRV00133,2024-06-01T01:00:00.000Z,New York,OH,85.4,3.269,279.17,FC612291,2026-09-28T04:04:31.137Z
FUEL00180741,TRIP00003001,TRK00092,null,2022-02-08T00:00:00.000Z,Portland,MO,83.6,3.937,329.13,FC140257,2026-09-28T04:04:31.137Z
FUEL00180742,TRIP00064359,TRK00106,DRV00010,2024-04-08T12:00:00.000Z,Houston,MI,60.7,3.413,207.17,FC864332,2026-09-28T04:04:31.137Z
FUEL00180743,TRIP00044980,TRK00091,DRV00019,2023-08-02T18:00:00.000Z,Los Angeles,OH,147.1,4.007,589.43,FC309209,2026-09-28T04:04:31.137Z
FUEL00180744,TRIP00063420,TRK00095,DRV00110,2024-03-27T17:00:00.000Z,Seattle,IN,63.9,3.985,254.64,FC938982,2026-09-28T04:04:31.137Z
FUEL00180745,TRIP00050301,TRK00113,DRV00005,2023-10-11T03:00:00.000Z,New York,FL,176.6,3.798,670.73,FC941614,2026-09-28T04:04:31.137Z


In [0]:
# Check important Fuel Purchase fields for missing values
fuel_purchases_silver.select(
    "fuel_purchase_id",
    "trip_id",
    "truck_id",
    "driver_id",
    "purchase_date"
).where(
    col("fuel_purchase_id").isNull() |
    col("trip_id").isNull() |
    col("truck_id").isNull()
).show(30, truncate=False)

+----------------+------------+--------+---------+-------------------+
|fuel_purchase_id|trip_id     |truck_id|driver_id|purchase_date      |
+----------------+------------+--------+---------+-------------------+
|FUEL00000012    |TRIP00079405|NULL    |DRV00085 |2024-10-16 19:00:00|
|FUEL00000038    |TRIP00036457|NULL    |DRV00020 |2023-04-12 11:00:00|
|FUEL00000112    |TRIP00081338|NULL    |DRV00142 |2024-11-08 11:00:00|
|FUEL00000234    |TRIP00058085|NULL    |DRV00087 |2024-01-19 18:00:00|
|FUEL00000353    |TRIP00038469|NULL    |DRV00037 |2023-05-11 05:00:00|
|FUEL00000379    |TRIP00068154|NULL    |DRV00112 |2024-05-27 00:00:00|
|FUEL00000391    |TRIP00049768|NULL    |DRV00034 |2023-10-04 19:00:00|
|FUEL00000407    |TRIP00032323|NULL    |DRV00054 |2023-02-18 00:00:00|
|FUEL00000425    |TRIP00083278|NULL    |DRV00118 |2024-12-05 13:00:00|
|FUEL00000469    |TRIP00013176|NULL    |DRV00056 |2022-06-20 14:00:00|
|FUEL00000495    |TRIP00071567|NULL    |DRV00113 |2024-07-09 13:00:00|
|FUEL0

In [0]:
# Write Fuel Purchases data to Silver using fuel_purchase_id as the key
write_to_silver(
    fuel_purchases_silver,
    "fuel_purchases",
    "target.fuel_purchase_id = source.fuel_purchase_id"
)

fuel_purchase_id,trip_id,truck_id,driver_id,purchase_date,location_city,location_state,gallons,price_per_gallon,total_cost,fuel_card_number,_ingested_at
FUEL00000001,TRIP00051284,TRK00045,DRV00102,2023-10-22T05:00:00.000Z,Columbus,MN,131.6,3.399,447.31,FC567161,2026-09-28T04:04:31.137Z
FUEL00000002,TRIP00073723,TRK00013,null,2024-08-04T08:00:00.000Z,New York,AZ,139.9,3.18,444.88,FC717910,2026-09-28T04:04:31.137Z
FUEL00000003,TRIP00018286,TRK00024,DRV00142,2022-08-23T13:00:00.000Z,Seattle,NE,189.3,3.804,720.1,FC912816,2026-09-28T04:04:31.137Z
FUEL00000004,TRIP00008465,TRK00018,DRV00047,2022-04-20T00:00:00.000Z,Memphis,NY,130.7,3.579,467.78,FC776357,2026-09-28T04:04:31.137Z
FUEL00000005,TRIP00069700,TRK00054,null,2024-06-13T09:00:00.000Z,Columbus,AZ,185.3,3.422,634.1,FC391024,2026-09-28T04:04:31.137Z
FUEL00000006,TRIP00062650,TRK00012,DRV00104,2024-03-18T23:00:00.000Z,Dallas,NY,182.2,3.494,636.61,FC277504,2026-09-28T04:04:31.137Z
FUEL00000007,TRIP00000851,TRK00072,DRV00127,2022-01-11T21:00:00.000Z,Miami,TX,146.3,3.914,572.62,FC256663,2026-09-28T04:04:31.137Z
FUEL00000008,TRIP00012033,TRK00073,DRV00084,2022-06-06T00:00:00.000Z,Phoenix,PA,113.3,4.471,506.56,FC543608,2026-09-28T04:04:31.137Z
FUEL00000009,TRIP00023596,TRK00101,DRV00080,2022-10-28T02:00:00.000Z,Miami,NC,127.0,4.445,564.51,FC484857,2026-09-28T04:04:31.137Z
FUEL00000010,TRIP00037859,TRK00085,DRV00002,2023-04-30T09:00:00.000Z,Dallas,MO,134.7,3.357,452.19,FC753360,2026-09-28T04:04:31.137Z


In [0]:
# Read Maintenance Records from the Bronze Delta layer
maintenance_records_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/maintenance_records")
)

# Display sample records
display(maintenance_records_bronze.limit(30))

maintenance_id,truck_id,maintenance_date,maintenance_type,odometer_reading,labor_hours,labor_cost,parts_cost,total_cost,facility_location,downtime_hours,service_description,_ingested_at
MAINT00000001,TRK00085,2022-01-01,Inspection,400255,7.8,781.42,10.41,791.83,Kansas City,22.2,Emergency Inspection,2026-09-28T04:04:40.437Z
MAINT00000002,TRK00041,2022-01-01,Tire,268041,0.9,80.91,3207.16,3288.07,Seattle,8.0,Scheduled Tire,2026-09-28T04:04:40.437Z
MAINT00000003,TRK00090,2022-01-01,Preventive,698915,7.2,733.18,1570.57,2303.75,Miami,16.5,Routine Preventive,2026-09-28T04:04:40.437Z
MAINT00000004,TRK00056,2022-01-02,Inspection,416522,2.8,298.8,67.48,366.28,Kansas City,34.6,Emergency Inspection,2026-09-28T04:04:40.437Z
MAINT00000005,TRK00029,2022-01-02,Repair,179968,4.6,409.08,982.59,1391.67,Omaha,4.4,Emergency Repair,2026-09-28T04:04:40.437Z
MAINT00000006,TRK00120,2022-01-02,Preventive,680507,5.0,498.12,3029.54,3527.66,Omaha,43.5,Scheduled Preventive,2026-09-28T04:04:40.437Z
MAINT00000007,TRK00017,2022-01-02,Repair,599078,0.9,83.38,1098.9,1182.28,Denver,34.8,Emergency Repair,2026-09-28T04:04:40.437Z
MAINT00000008,TRK00038,2022-01-03,Tire,605379,7.1,785.79,1483.25,2269.04,Miami,3.9,Scheduled Tire,2026-09-28T04:04:40.437Z
MAINT00000009,TRK00054,2022-01-03,Inspection,232147,4.0,371.51,143.21,514.72,Dallas,14.0,Routine Inspection,2026-09-28T04:04:40.437Z
MAINT00000010,TRK00076,2022-01-03,Inspection,222428,0.6,74.65,14.12,88.77,Columbus,23.0,Emergency Inspection,2026-09-28T04:04:40.437Z


In [0]:
# Clean and standardize the Maintenance Records data
maintenance_records_silver = (
    maintenance_records_bronze

    # Remove extra spaces from text columns
    .withColumn("maintenance_id", trim(col("maintenance_id")))
    .withColumn("truck_id", trim(col("truck_id")))
    .withColumn("maintenance_type", trim(col("maintenance_type")))
    .withColumn("facility_location", trim(col("facility_location")))
    .withColumn("service_description", trim(col("service_description")))

    # Standardize maintenance type
    .withColumn("maintenance_type", upper(col("maintenance_type")))
)

# Display cleaned Maintenance Records data
display(maintenance_records_silver.limit(30))

maintenance_id,truck_id,maintenance_date,maintenance_type,odometer_reading,labor_hours,labor_cost,parts_cost,total_cost,facility_location,downtime_hours,service_description,_ingested_at
MAINT00000001,TRK00085,2022-01-01,INSPECTION,400255,7.8,781.42,10.41,791.83,Kansas City,22.2,Emergency Inspection,2026-09-28T04:04:40.437Z
MAINT00000002,TRK00041,2022-01-01,TIRE,268041,0.9,80.91,3207.16,3288.07,Seattle,8.0,Scheduled Tire,2026-09-28T04:04:40.437Z
MAINT00000003,TRK00090,2022-01-01,PREVENTIVE,698915,7.2,733.18,1570.57,2303.75,Miami,16.5,Routine Preventive,2026-09-28T04:04:40.437Z
MAINT00000004,TRK00056,2022-01-02,INSPECTION,416522,2.8,298.8,67.48,366.28,Kansas City,34.6,Emergency Inspection,2026-09-28T04:04:40.437Z
MAINT00000005,TRK00029,2022-01-02,REPAIR,179968,4.6,409.08,982.59,1391.67,Omaha,4.4,Emergency Repair,2026-09-28T04:04:40.437Z
MAINT00000006,TRK00120,2022-01-02,PREVENTIVE,680507,5.0,498.12,3029.54,3527.66,Omaha,43.5,Scheduled Preventive,2026-09-28T04:04:40.437Z
MAINT00000007,TRK00017,2022-01-02,REPAIR,599078,0.9,83.38,1098.9,1182.28,Denver,34.8,Emergency Repair,2026-09-28T04:04:40.437Z
MAINT00000008,TRK00038,2022-01-03,TIRE,605379,7.1,785.79,1483.25,2269.04,Miami,3.9,Scheduled Tire,2026-09-28T04:04:40.437Z
MAINT00000009,TRK00054,2022-01-03,INSPECTION,232147,4.0,371.51,143.21,514.72,Dallas,14.0,Routine Inspection,2026-09-28T04:04:40.437Z
MAINT00000010,TRK00076,2022-01-03,INSPECTION,222428,0.6,74.65,14.12,88.77,Columbus,23.0,Emergency Inspection,2026-09-28T04:04:40.437Z


In [0]:
# Check important Maintenance fields for missing values
maintenance_records_silver.select(
    "maintenance_id",
    "truck_id",
    "maintenance_date",
    "maintenance_type"
).where(
    col("maintenance_id").isNull() |
    col("truck_id").isNull()
).show(30, truncate=False)

+--------------+--------+----------------+----------------+
|maintenance_id|truck_id|maintenance_date|maintenance_type|
+--------------+--------+----------------+----------------+
+--------------+--------+----------------+----------------+



In [0]:
# Write Maintenance Records data to Silver using maintenance_id as the key
write_to_silver(
    maintenance_records_silver,
    "maintenance_records",
    "target.maintenance_id = source.maintenance_id"
)

maintenance_id,truck_id,maintenance_date,maintenance_type,odometer_reading,labor_hours,labor_cost,parts_cost,total_cost,facility_location,downtime_hours,service_description,_ingested_at
MAINT00000001,TRK00085,2022-01-01,INSPECTION,400255,7.8,781.42,10.41,791.83,Kansas City,22.2,Emergency Inspection,2026-09-28T04:04:40.437Z
MAINT00000002,TRK00041,2022-01-01,TIRE,268041,0.9,80.91,3207.16,3288.07,Seattle,8.0,Scheduled Tire,2026-09-28T04:04:40.437Z
MAINT00000003,TRK00090,2022-01-01,PREVENTIVE,698915,7.2,733.18,1570.57,2303.75,Miami,16.5,Routine Preventive,2026-09-28T04:04:40.437Z
MAINT00000004,TRK00056,2022-01-02,INSPECTION,416522,2.8,298.8,67.48,366.28,Kansas City,34.6,Emergency Inspection,2026-09-28T04:04:40.437Z
MAINT00000005,TRK00029,2022-01-02,REPAIR,179968,4.6,409.08,982.59,1391.67,Omaha,4.4,Emergency Repair,2026-09-28T04:04:40.437Z
MAINT00000006,TRK00120,2022-01-02,PREVENTIVE,680507,5.0,498.12,3029.54,3527.66,Omaha,43.5,Scheduled Preventive,2026-09-28T04:04:40.437Z
MAINT00000007,TRK00017,2022-01-02,REPAIR,599078,0.9,83.38,1098.9,1182.28,Denver,34.8,Emergency Repair,2026-09-28T04:04:40.437Z
MAINT00000008,TRK00038,2022-01-03,TIRE,605379,7.1,785.79,1483.25,2269.04,Miami,3.9,Scheduled Tire,2026-09-28T04:04:40.437Z
MAINT00000009,TRK00054,2022-01-03,INSPECTION,232147,4.0,371.51,143.21,514.72,Dallas,14.0,Routine Inspection,2026-09-28T04:04:40.437Z
MAINT00000010,TRK00076,2022-01-03,INSPECTION,222428,0.6,74.65,14.12,88.77,Columbus,23.0,Emergency Inspection,2026-09-28T04:04:40.437Z


In [0]:
# Read Delivery Events from the Bronze Delta layer
delivery_events_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/delivery_events")
)

# Display sample records
display(delivery_events_bronze.limit(30))

event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,on_time_flag,location_city,location_state,_ingested_at
EVT00111800,LOAD00055900,TRIP00055900,Delivery,FAC00011,2023-12-20T23:40:27.864Z,2023-12-20T23:58:52.594Z,0,true,Kansas City,MO,2026-09-28T04:04:49.217Z
EVT00111801,LOAD00055901,TRIP00055901,Pickup,FAC00025,2023-12-20T13:00:00.000Z,2023-12-20T13:09:07.273Z,2,true,New York,NY,2026-09-28T04:04:49.217Z
EVT00111802,LOAD00055901,TRIP00055901,Delivery,FAC00017,2023-12-20T23:39:07.273Z,2023-12-20T22:11:05.325Z,109,true,Columbus,OH,2026-09-28T04:04:49.217Z
EVT00111803,LOAD00055902,TRIP00055902,Pickup,FAC00039,2023-12-20T07:00:00.000Z,2023-12-20T09:56:28.921Z,0,false,Memphis,TN,2026-09-28T04:04:49.217Z
EVT00111804,LOAD00055902,TRIP00055902,Delivery,FAC00014,2023-12-21T18:08:28.921Z,2023-12-21T20:59:48.790Z,146,false,Los Angeles,CA,2026-09-28T04:04:49.217Z
EVT00111805,LOAD00055903,TRIP00055903,Pickup,FAC00029,2023-12-20T08:00:00.000Z,2023-12-20T06:29:26.762Z,23,true,Las Vegas,NV,2026-09-28T04:04:49.217Z
EVT00111806,LOAD00055903,TRIP00055903,Delivery,FAC00012,2023-12-21T03:35:26.762Z,2023-12-21T04:49:44.468Z,228,true,Dallas,TX,2026-09-28T04:04:49.217Z
EVT00111807,LOAD00055904,TRIP00055904,Pickup,FAC00046,2023-12-20T11:00:00.000Z,2023-12-20T11:48:21.034Z,0,true,Seattle,WA,2026-09-28T04:04:49.217Z
EVT00111808,LOAD00055904,TRIP00055904,Delivery,FAC00049,2023-12-22T03:48:21.034Z,2023-12-22T05:04:08.701Z,120,true,Indianapolis,IN,2026-09-28T04:04:49.217Z
EVT00111809,LOAD00055905,TRIP00055905,Pickup,FAC00029,2023-12-20T06:00:00.000Z,2023-12-20T09:05:50.030Z,117,false,Columbus,OH,2026-09-28T04:04:49.217Z


In [0]:
# Clean and standardize the Delivery Events data
delivery_events_silver = (
    delivery_events_bronze

    # Remove extra spaces from ID and location columns
    .withColumn("event_id", trim(col("event_id")))
    .withColumn("load_id", trim(col("load_id")))
    .withColumn("trip_id", trim(col("trip_id")))
    .withColumn("event_type", trim(col("event_type")))
    .withColumn("facility_id", trim(col("facility_id")))
    .withColumn("location_city", trim(col("location_city")))
    .withColumn("location_state", trim(col("location_state")))

    # Standardize categorical values
    .withColumn("event_type", upper(col("event_type")))
    .withColumn("location_state", upper(col("location_state")))
)

# Display cleaned Delivery Events data
display(delivery_events_silver.limit(30))

event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,on_time_flag,location_city,location_state,_ingested_at
EVT00111800,LOAD00055900,TRIP00055900,DELIVERY,FAC00011,2023-12-20T23:40:27.864Z,2023-12-20T23:58:52.594Z,0,true,Kansas City,MO,2026-09-28T04:04:49.217Z
EVT00111801,LOAD00055901,TRIP00055901,PICKUP,FAC00025,2023-12-20T13:00:00.000Z,2023-12-20T13:09:07.273Z,2,true,New York,NY,2026-09-28T04:04:49.217Z
EVT00111802,LOAD00055901,TRIP00055901,DELIVERY,FAC00017,2023-12-20T23:39:07.273Z,2023-12-20T22:11:05.325Z,109,true,Columbus,OH,2026-09-28T04:04:49.217Z
EVT00111803,LOAD00055902,TRIP00055902,PICKUP,FAC00039,2023-12-20T07:00:00.000Z,2023-12-20T09:56:28.921Z,0,false,Memphis,TN,2026-09-28T04:04:49.217Z
EVT00111804,LOAD00055902,TRIP00055902,DELIVERY,FAC00014,2023-12-21T18:08:28.921Z,2023-12-21T20:59:48.790Z,146,false,Los Angeles,CA,2026-09-28T04:04:49.217Z
EVT00111805,LOAD00055903,TRIP00055903,PICKUP,FAC00029,2023-12-20T08:00:00.000Z,2023-12-20T06:29:26.762Z,23,true,Las Vegas,NV,2026-09-28T04:04:49.217Z
EVT00111806,LOAD00055903,TRIP00055903,DELIVERY,FAC00012,2023-12-21T03:35:26.762Z,2023-12-21T04:49:44.468Z,228,true,Dallas,TX,2026-09-28T04:04:49.217Z
EVT00111807,LOAD00055904,TRIP00055904,PICKUP,FAC00046,2023-12-20T11:00:00.000Z,2023-12-20T11:48:21.034Z,0,true,Seattle,WA,2026-09-28T04:04:49.217Z
EVT00111808,LOAD00055904,TRIP00055904,DELIVERY,FAC00049,2023-12-22T03:48:21.034Z,2023-12-22T05:04:08.701Z,120,true,Indianapolis,IN,2026-09-28T04:04:49.217Z
EVT00111809,LOAD00055905,TRIP00055905,PICKUP,FAC00029,2023-12-20T06:00:00.000Z,2023-12-20T09:05:50.030Z,117,false,Columbus,OH,2026-09-28T04:04:49.217Z


In [0]:
# Check important Delivery Event fields for missing values
delivery_events_silver.select(
    "event_id",
    "load_id",
    "trip_id",
    "event_type",
    "scheduled_datetime",
    "actual_datetime"
).where(
    col("event_id").isNull() |
    col("load_id").isNull() |
    col("trip_id").isNull()
).show(30, truncate=False)

+--------+-------+-------+----------+------------------+---------------+
|event_id|load_id|trip_id|event_type|scheduled_datetime|actual_datetime|
+--------+-------+-------+----------+------------------+---------------+
+--------+-------+-------+----------+------------------+---------------+



In [0]:
# Write Delivery Events data to Silver using event_id as the key
write_to_silver(
    delivery_events_silver,
    "delivery_events",
    "target.event_id = source.event_id"
)

event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,on_time_flag,location_city,location_state,_ingested_at
EVT00111800,LOAD00055900,TRIP00055900,DELIVERY,FAC00011,2023-12-20T23:40:27.864Z,2023-12-20T23:58:52.594Z,0,true,Kansas City,MO,2026-09-28T04:04:49.217Z
EVT00111801,LOAD00055901,TRIP00055901,PICKUP,FAC00025,2023-12-20T13:00:00.000Z,2023-12-20T13:09:07.273Z,2,true,New York,NY,2026-09-28T04:04:49.217Z
EVT00111802,LOAD00055901,TRIP00055901,DELIVERY,FAC00017,2023-12-20T23:39:07.273Z,2023-12-20T22:11:05.325Z,109,true,Columbus,OH,2026-09-28T04:04:49.217Z
EVT00111803,LOAD00055902,TRIP00055902,PICKUP,FAC00039,2023-12-20T07:00:00.000Z,2023-12-20T09:56:28.921Z,0,false,Memphis,TN,2026-09-28T04:04:49.217Z
EVT00111804,LOAD00055902,TRIP00055902,DELIVERY,FAC00014,2023-12-21T18:08:28.921Z,2023-12-21T20:59:48.790Z,146,false,Los Angeles,CA,2026-09-28T04:04:49.217Z
EVT00111805,LOAD00055903,TRIP00055903,PICKUP,FAC00029,2023-12-20T08:00:00.000Z,2023-12-20T06:29:26.762Z,23,true,Las Vegas,NV,2026-09-28T04:04:49.217Z
EVT00111806,LOAD00055903,TRIP00055903,DELIVERY,FAC00012,2023-12-21T03:35:26.762Z,2023-12-21T04:49:44.468Z,228,true,Dallas,TX,2026-09-28T04:04:49.217Z
EVT00111807,LOAD00055904,TRIP00055904,PICKUP,FAC00046,2023-12-20T11:00:00.000Z,2023-12-20T11:48:21.034Z,0,true,Seattle,WA,2026-09-28T04:04:49.217Z
EVT00111808,LOAD00055904,TRIP00055904,DELIVERY,FAC00049,2023-12-22T03:48:21.034Z,2023-12-22T05:04:08.701Z,120,true,Indianapolis,IN,2026-09-28T04:04:49.217Z
EVT00111809,LOAD00055905,TRIP00055905,PICKUP,FAC00029,2023-12-20T06:00:00.000Z,2023-12-20T09:05:50.030Z,117,false,Columbus,OH,2026-09-28T04:04:49.217Z


In [0]:
# Read Safety Incidents from the Bronze Delta layer
safety_incidents_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/safety_incidents")
)

# Display sample records
display(safety_incidents_bronze.limit(30))

incident_id,trip_id,truck_id,driver_id,incident_date,incident_type,location_city,location_state,at_fault_flag,injury_flag,vehicle_damage_cost,cargo_damage_cost,claim_amount,preventable_flag,description,_ingested_at
INC00000001,TRIP00036079,TRK00006,DRV00006,2023-04-09T14:00:00.000Z,Moving Violation,Columbus,PA,true,false,12629.26,0.0,12629.26,true,Severe incident involving equipment,2026-09-28T04:04:59.231Z
INC00000002,TRIP00032462,TRK00084,DRV00006,2023-02-19T11:00:00.000Z,Moving Violation,Columbus,NC,false,false,2700.7,14284.24,16984.94,false,Severe incident involving weather,2026-09-28T04:04:59.231Z
INC00000003,TRIP00067583,TRK00106,DRV00119,2024-05-20T01:00:00.000Z,Customer Complaint,Seattle,OK,true,false,24302.32,0.0,24302.32,true,Minor incident involving traffic,2026-09-28T04:04:59.231Z
INC00000004,TRIP00049128,TRK00085,DRV00134,2023-09-26T03:00:00.000Z,Equipment Damage,Kansas City,TN,true,false,4879.13,0.0,4879.13,true,Moderate incident involving equipment,2026-09-28T04:04:59.231Z
INC00000005,TRIP00062884,TRK00014,DRV00133,2024-03-20T14:00:00.000Z,DOT Violation,Las Vegas,IN,false,false,14951.4,0.0,14951.4,false,Minor incident involving other driver,2026-09-28T04:04:59.231Z
INC00000006,TRIP00004361,TRK00007,DRV00005,2022-02-27T18:00:00.000Z,Equipment Damage,Oklahoma City,CO,false,true,0.0,0.0,0.0,false,Minor incident involving weather,2026-09-28T04:04:59.231Z
INC00000007,TRIP00012773,TRK00007,DRV00109,2022-06-13T02:00:00.000Z,DOT Violation,Portland,TX,true,false,12589.05,28315.16,40904.21,true,Moderate incident involving traffic,2026-09-28T04:04:59.231Z
INC00000008,TRIP00072329,TRK00115,DRV00060,2024-07-17T20:00:00.000Z,Accident,Phoenix,TX,false,false,15572.01,0.0,15572.01,false,Moderate incident involving equipment,2026-09-28T04:04:59.231Z
INC00000009,TRIP00043143,TRK00107,DRV00003,2023-07-08T02:00:00.000Z,Customer Complaint,Oklahoma City,TX,false,true,22283.17,379.1,22662.27,false,Moderate incident involving traffic,2026-09-28T04:04:59.231Z
INC00000010,TRIP00068700,TRK00074,DRV00132,2024-06-03T04:00:00.000Z,Equipment Damage,Detroit,PA,false,false,0.0,33901.9,33901.9,false,Severe incident involving equipment,2026-09-28T04:04:59.231Z


In [0]:
# Clean and standardize the Safety Incidents data
safety_incidents_silver = (
    safety_incidents_bronze

    # Remove extra spaces from ID and location columns
    .withColumn("incident_id", trim(col("incident_id")))
    .withColumn("trip_id", trim(col("trip_id")))
    .withColumn("truck_id", trim(col("truck_id")))
    .withColumn("driver_id", trim(col("driver_id")))
    .withColumn("incident_type", trim(col("incident_type")))
    .withColumn("location_city", trim(col("location_city")))
    .withColumn("location_state", trim(col("location_state")))
    .withColumn("description", trim(col("description")))

    # Standardize categorical values
    .withColumn("incident_type", upper(col("incident_type")))
    .withColumn("location_state", upper(col("location_state")))
)

# Display cleaned Safety Incidents data
display(safety_incidents_silver.limit(30))

incident_id,trip_id,truck_id,driver_id,incident_date,incident_type,location_city,location_state,at_fault_flag,injury_flag,vehicle_damage_cost,cargo_damage_cost,claim_amount,preventable_flag,description,_ingested_at
INC00000001,TRIP00036079,TRK00006,DRV00006,2023-04-09T14:00:00.000Z,MOVING VIOLATION,Columbus,PA,true,false,12629.26,0.0,12629.26,true,Severe incident involving equipment,2026-09-28T04:04:59.231Z
INC00000002,TRIP00032462,TRK00084,DRV00006,2023-02-19T11:00:00.000Z,MOVING VIOLATION,Columbus,NC,false,false,2700.7,14284.24,16984.94,false,Severe incident involving weather,2026-09-28T04:04:59.231Z
INC00000003,TRIP00067583,TRK00106,DRV00119,2024-05-20T01:00:00.000Z,CUSTOMER COMPLAINT,Seattle,OK,true,false,24302.32,0.0,24302.32,true,Minor incident involving traffic,2026-09-28T04:04:59.231Z
INC00000004,TRIP00049128,TRK00085,DRV00134,2023-09-26T03:00:00.000Z,EQUIPMENT DAMAGE,Kansas City,TN,true,false,4879.13,0.0,4879.13,true,Moderate incident involving equipment,2026-09-28T04:04:59.231Z
INC00000005,TRIP00062884,TRK00014,DRV00133,2024-03-20T14:00:00.000Z,DOT VIOLATION,Las Vegas,IN,false,false,14951.4,0.0,14951.4,false,Minor incident involving other driver,2026-09-28T04:04:59.231Z
INC00000006,TRIP00004361,TRK00007,DRV00005,2022-02-27T18:00:00.000Z,EQUIPMENT DAMAGE,Oklahoma City,CO,false,true,0.0,0.0,0.0,false,Minor incident involving weather,2026-09-28T04:04:59.231Z
INC00000007,TRIP00012773,TRK00007,DRV00109,2022-06-13T02:00:00.000Z,DOT VIOLATION,Portland,TX,true,false,12589.05,28315.16,40904.21,true,Moderate incident involving traffic,2026-09-28T04:04:59.231Z
INC00000008,TRIP00072329,TRK00115,DRV00060,2024-07-17T20:00:00.000Z,ACCIDENT,Phoenix,TX,false,false,15572.01,0.0,15572.01,false,Moderate incident involving equipment,2026-09-28T04:04:59.231Z
INC00000009,TRIP00043143,TRK00107,DRV00003,2023-07-08T02:00:00.000Z,CUSTOMER COMPLAINT,Oklahoma City,TX,false,true,22283.17,379.1,22662.27,false,Moderate incident involving traffic,2026-09-28T04:04:59.231Z
INC00000010,TRIP00068700,TRK00074,DRV00132,2024-06-03T04:00:00.000Z,EQUIPMENT DAMAGE,Detroit,PA,false,false,0.0,33901.9,33901.9,false,Severe incident involving equipment,2026-09-28T04:04:59.231Z


In [0]:
# Check important Safety Incident fields for missing values
safety_incidents_silver.select(
    "incident_id",
    "trip_id",
    "truck_id",
    "driver_id",
    "incident_date"
).where(
    col("incident_id").isNull() |
    col("trip_id").isNull() |
    col("truck_id").isNull() |
    col("driver_id").isNull()
).show(30, truncate=False)

+-----------+------------+--------+---------+-------------------+
|incident_id|trip_id     |truck_id|driver_id|incident_date      |
+-----------+------------+--------+---------+-------------------+
|INC00000060|TRIP00067707|NULL    |DRV00101 |2024-05-21 07:00:00|
|INC00000086|TRIP00078852|TRK00120|NULL     |2024-10-10 15:00:00|
+-----------+------------+--------+---------+-------------------+



In [0]:
# Write Safety Incidents data to Silver using incident_id as the key
write_to_silver(
    safety_incidents_silver,
    "safety_incidents",
    "target.incident_id = source.incident_id"
)

incident_id,trip_id,truck_id,driver_id,incident_date,incident_type,location_city,location_state,at_fault_flag,injury_flag,vehicle_damage_cost,cargo_damage_cost,claim_amount,preventable_flag,description,_ingested_at
INC00000001,TRIP00036079,TRK00006,DRV00006,2023-04-09T14:00:00.000Z,MOVING VIOLATION,Columbus,PA,true,false,12629.26,0.0,12629.26,true,Severe incident involving equipment,2026-09-28T04:04:59.231Z
INC00000002,TRIP00032462,TRK00084,DRV00006,2023-02-19T11:00:00.000Z,MOVING VIOLATION,Columbus,NC,false,false,2700.7,14284.24,16984.94,false,Severe incident involving weather,2026-09-28T04:04:59.231Z
INC00000003,TRIP00067583,TRK00106,DRV00119,2024-05-20T01:00:00.000Z,CUSTOMER COMPLAINT,Seattle,OK,true,false,24302.32,0.0,24302.32,true,Minor incident involving traffic,2026-09-28T04:04:59.231Z
INC00000004,TRIP00049128,TRK00085,DRV00134,2023-09-26T03:00:00.000Z,EQUIPMENT DAMAGE,Kansas City,TN,true,false,4879.13,0.0,4879.13,true,Moderate incident involving equipment,2026-09-28T04:04:59.231Z
INC00000005,TRIP00062884,TRK00014,DRV00133,2024-03-20T14:00:00.000Z,DOT VIOLATION,Las Vegas,IN,false,false,14951.4,0.0,14951.4,false,Minor incident involving other driver,2026-09-28T04:04:59.231Z
INC00000006,TRIP00004361,TRK00007,DRV00005,2022-02-27T18:00:00.000Z,EQUIPMENT DAMAGE,Oklahoma City,CO,false,true,0.0,0.0,0.0,false,Minor incident involving weather,2026-09-28T04:04:59.231Z
INC00000007,TRIP00012773,TRK00007,DRV00109,2022-06-13T02:00:00.000Z,DOT VIOLATION,Portland,TX,true,false,12589.05,28315.16,40904.21,true,Moderate incident involving traffic,2026-09-28T04:04:59.231Z
INC00000008,TRIP00072329,TRK00115,DRV00060,2024-07-17T20:00:00.000Z,ACCIDENT,Phoenix,TX,false,false,15572.01,0.0,15572.01,false,Moderate incident involving equipment,2026-09-28T04:04:59.231Z
INC00000009,TRIP00043143,TRK00107,DRV00003,2023-07-08T02:00:00.000Z,CUSTOMER COMPLAINT,Oklahoma City,TX,false,true,22283.17,379.1,22662.27,false,Moderate incident involving traffic,2026-09-28T04:04:59.231Z
INC00000010,TRIP00068700,TRK00074,DRV00132,2024-06-03T04:00:00.000Z,EQUIPMENT DAMAGE,Detroit,PA,false,false,0.0,33901.9,33901.9,false,Severe incident involving equipment,2026-09-28T04:04:59.231Z


In [0]:
# Read Driver Monthly Metrics from the Bronze Delta layer
driver_monthly_metrics_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/driver_monthly_metrics")
)

# Display sample records
display(driver_monthly_metrics_bronze.limit(30))

driver_id,month,trips_completed,total_miles,total_revenue,average_mpg,total_fuel_gallons,on_time_delivery_rate,average_idle_hours,_ingested_at
DRV00001,2022-01-01,26,36620,79141.59,6.61,5574.7,0.385,8.2,2026-09-28T04:05:07.101Z
DRV00001,2022-02-01,9,13515,27133.87,6.69,2095.1,0.333,6.8,2026-09-28T04:05:07.101Z
DRV00001,2022-03-01,20,30361,62399.62,6.36,4792.2,0.55,7.5,2026-09-28T04:05:07.101Z
DRV00001,2022-04-01,19,28127,68161.13,6.61,4251.4,0.474,8.4,2026-09-28T04:05:07.101Z
DRV00001,2022-05-01,14,16071,36256.82,6.35,2573.1,0.214,6.8,2026-09-28T04:05:07.101Z
DRV00001,2022-06-01,13,17291,39709.91,6.38,2767.8,0.615,5.5,2026-09-28T04:05:07.101Z
DRV00001,2022-07-01,21,32772,66855.24,6.41,5198.6,0.476,7.5,2026-09-28T04:05:07.101Z
DRV00001,2022-08-01,20,27010,58514.48,6.35,4331.9,0.45,5.8,2026-09-28T04:05:07.101Z
DRV00001,2022-09-01,20,30866,72027.99,6.21,4984.9,0.55,5.8,2026-09-28T04:05:07.101Z
DRV00001,2022-10-01,12,23943,53495.8,6.26,3832.4,0.333,7.0,2026-09-28T04:05:07.101Z


In [0]:
# Clean and standardize Driver Monthly Metrics
driver_monthly_metrics_silver = (
    driver_monthly_metrics_bronze

    # Remove extra spaces from the driver ID
    .withColumn("driver_id", trim(col("driver_id")))
)

# Display cleaned records
display(driver_monthly_metrics_silver.limit(30))

driver_id,month,trips_completed,total_miles,total_revenue,average_mpg,total_fuel_gallons,on_time_delivery_rate,average_idle_hours,_ingested_at
DRV00001,2022-01-01,26,36620,79141.59,6.61,5574.7,0.385,8.2,2026-09-28T04:05:07.101Z
DRV00001,2022-02-01,9,13515,27133.87,6.69,2095.1,0.333,6.8,2026-09-28T04:05:07.101Z
DRV00001,2022-03-01,20,30361,62399.62,6.36,4792.2,0.55,7.5,2026-09-28T04:05:07.101Z
DRV00001,2022-04-01,19,28127,68161.13,6.61,4251.4,0.474,8.4,2026-09-28T04:05:07.101Z
DRV00001,2022-05-01,14,16071,36256.82,6.35,2573.1,0.214,6.8,2026-09-28T04:05:07.101Z
DRV00001,2022-06-01,13,17291,39709.91,6.38,2767.8,0.615,5.5,2026-09-28T04:05:07.101Z
DRV00001,2022-07-01,21,32772,66855.24,6.41,5198.6,0.476,7.5,2026-09-28T04:05:07.101Z
DRV00001,2022-08-01,20,27010,58514.48,6.35,4331.9,0.45,5.8,2026-09-28T04:05:07.101Z
DRV00001,2022-09-01,20,30866,72027.99,6.21,4984.9,0.55,5.8,2026-09-28T04:05:07.101Z
DRV00001,2022-10-01,12,23943,53495.8,6.26,3832.4,0.333,7.0,2026-09-28T04:05:07.101Z


In [0]:
# Check the composite key fields for missing values
driver_monthly_metrics_silver.select(
    "driver_id",
    "month"
).where(
    col("driver_id").isNull() |
    col("month").isNull()
).show(30, truncate=False)

+---------+-----+
|driver_id|month|
+---------+-----+
+---------+-----+



In [0]:
# Write Driver Monthly Metrics using driver_id and month as the composite key
write_to_silver(
    driver_monthly_metrics_silver,
    "driver_monthly_metrics",
    """
    target.driver_id = source.driver_id
    AND target.month = source.month
    """
)

driver_id,month,trips_completed,total_miles,total_revenue,average_mpg,total_fuel_gallons,on_time_delivery_rate,average_idle_hours,_ingested_at
DRV00001,2022-01-01,26,36620,79141.59,6.61,5574.7,0.385,8.2,2026-09-28T04:05:07.101Z
DRV00001,2022-02-01,9,13515,27133.87,6.69,2095.1,0.333,6.8,2026-09-28T04:05:07.101Z
DRV00001,2022-03-01,20,30361,62399.62,6.36,4792.2,0.55,7.5,2026-09-28T04:05:07.101Z
DRV00001,2022-04-01,19,28127,68161.13,6.61,4251.4,0.474,8.4,2026-09-28T04:05:07.101Z
DRV00001,2022-05-01,14,16071,36256.82,6.35,2573.1,0.214,6.8,2026-09-28T04:05:07.101Z
DRV00001,2022-06-01,13,17291,39709.91,6.38,2767.8,0.615,5.5,2026-09-28T04:05:07.101Z
DRV00001,2022-07-01,21,32772,66855.24,6.41,5198.6,0.476,7.5,2026-09-28T04:05:07.101Z
DRV00001,2022-08-01,20,27010,58514.48,6.35,4331.9,0.45,5.8,2026-09-28T04:05:07.101Z
DRV00001,2022-09-01,20,30866,72027.99,6.21,4984.9,0.55,5.8,2026-09-28T04:05:07.101Z
DRV00001,2022-10-01,12,23943,53495.8,6.26,3832.4,0.333,7.0,2026-09-28T04:05:07.101Z


In [0]:
# Read Truck Utilization Metrics from the Bronze Delta layer
truck_utilization_metrics_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/truck_utilization_metrics")
)

# Display sample records
display(truck_utilization_metrics_bronze.limit(30))

truck_id,month,trips_completed,total_miles,total_revenue,average_mpg,maintenance_events,maintenance_cost,downtime_hours,utilization_rate,_ingested_at
TRK00001,2022-01-01,22,39269,84792.02,6.78,2,4380.98,63.1,0.71,2026-09-28T04:05:16.678Z
TRK00001,2022-02-01,27,40787,88809.05,6.23,0,0.0,0.0,0.964,2026-09-28T04:05:16.678Z
TRK00001,2022-03-01,34,57422,120453.71,6.49,0,0.0,0.0,1.097,2026-09-28T04:05:16.678Z
TRK00001,2022-04-01,20,34428,73531.28,6.5,2,3548.15,71.8,0.667,2026-09-28T04:05:16.678Z
TRK00001,2022-05-01,16,20851,40791.53,6.38,1,1969.42,43.9,0.516,2026-09-28T04:05:16.678Z
TRK00001,2022-06-01,31,47085,101659.37,6.46,0,0.0,0.0,1.033,2026-09-28T04:05:16.678Z
TRK00001,2022-07-01,28,38038,84143.99,6.54,0,0.0,0.0,0.903,2026-09-28T04:05:16.678Z
TRK00001,2022-08-01,27,35902,82253.78,6.3,1,382.57,20.4,0.871,2026-09-28T04:05:16.678Z
TRK00001,2022-09-01,24,40898,88581.76,6.56,1,3372.41,46.5,0.8,2026-09-28T04:05:16.678Z
TRK00001,2022-10-01,37,51364,108130.03,6.42,2,1744.63,58.7,1.194,2026-09-28T04:05:16.678Z


In [0]:
# Clean and standardize Truck Utilization Metrics
truck_utilization_metrics_silver = (
    truck_utilization_metrics_bronze

    # Remove extra spaces from the truck ID
    .withColumn("truck_id", trim(col("truck_id")))
)

# Display cleaned records
display(truck_utilization_metrics_silver.limit(30))

truck_id,month,trips_completed,total_miles,total_revenue,average_mpg,maintenance_events,maintenance_cost,downtime_hours,utilization_rate,_ingested_at
TRK00001,2022-01-01,22,39269,84792.02,6.78,2,4380.98,63.1,0.71,2026-09-28T04:05:16.678Z
TRK00001,2022-02-01,27,40787,88809.05,6.23,0,0.0,0.0,0.964,2026-09-28T04:05:16.678Z
TRK00001,2022-03-01,34,57422,120453.71,6.49,0,0.0,0.0,1.097,2026-09-28T04:05:16.678Z
TRK00001,2022-04-01,20,34428,73531.28,6.5,2,3548.15,71.8,0.667,2026-09-28T04:05:16.678Z
TRK00001,2022-05-01,16,20851,40791.53,6.38,1,1969.42,43.9,0.516,2026-09-28T04:05:16.678Z
TRK00001,2022-06-01,31,47085,101659.37,6.46,0,0.0,0.0,1.033,2026-09-28T04:05:16.678Z
TRK00001,2022-07-01,28,38038,84143.99,6.54,0,0.0,0.0,0.903,2026-09-28T04:05:16.678Z
TRK00001,2022-08-01,27,35902,82253.78,6.3,1,382.57,20.4,0.871,2026-09-28T04:05:16.678Z
TRK00001,2022-09-01,24,40898,88581.76,6.56,1,3372.41,46.5,0.8,2026-09-28T04:05:16.678Z
TRK00001,2022-10-01,37,51364,108130.03,6.42,2,1744.63,58.7,1.194,2026-09-28T04:05:16.678Z


In [0]:
# Check the composite key fields for missing values
truck_utilization_metrics_silver.select(
    "truck_id",
    "month"
).where(
    col("truck_id").isNull() |
    col("month").isNull()
).show(30, truncate=False)

+--------+-----+
|truck_id|month|
+--------+-----+
+--------+-----+



In [0]:
# Write Truck Utilization Metrics using truck_id and month as the composite key
write_to_silver(
    truck_utilization_metrics_silver,
    "truck_utilization_metrics",
    """
    target.truck_id = source.truck_id
    AND target.month = source.month
    """
)

truck_id,month,trips_completed,total_miles,total_revenue,average_mpg,maintenance_events,maintenance_cost,downtime_hours,utilization_rate,_ingested_at
TRK00001,2022-01-01,22,39269,84792.02,6.78,2,4380.98,63.1,0.71,2026-09-28T04:05:16.678Z
TRK00001,2022-02-01,27,40787,88809.05,6.23,0,0.0,0.0,0.964,2026-09-28T04:05:16.678Z
TRK00001,2022-03-01,34,57422,120453.71,6.49,0,0.0,0.0,1.097,2026-09-28T04:05:16.678Z
TRK00001,2022-04-01,20,34428,73531.28,6.5,2,3548.15,71.8,0.667,2026-09-28T04:05:16.678Z
TRK00001,2022-05-01,16,20851,40791.53,6.38,1,1969.42,43.9,0.516,2026-09-28T04:05:16.678Z
TRK00001,2022-06-01,31,47085,101659.37,6.46,0,0.0,0.0,1.033,2026-09-28T04:05:16.678Z
TRK00001,2022-07-01,28,38038,84143.99,6.54,0,0.0,0.0,0.903,2026-09-28T04:05:16.678Z
TRK00001,2022-08-01,27,35902,82253.78,6.3,1,382.57,20.4,0.871,2026-09-28T04:05:16.678Z
TRK00001,2022-09-01,24,40898,88581.76,6.56,1,3372.41,46.5,0.8,2026-09-28T04:05:16.678Z
TRK00001,2022-10-01,37,51364,108130.03,6.42,2,1744.63,58.7,1.194,2026-09-28T04:05:16.678Z


In [0]:
# Compare row counts between Bronze and Silver for all 14 tables

tables = [
    "customers",
    "drivers",
    "trucks",
    "trailers",
    "facilities",
    "routes",
    "loads",
    "trips",
    "fuel_purchases",
    "maintenance_records",
    "delivery_events",
    "safety_incidents",
    "driver_monthly_metrics",
    "truck_utilization_metrics"
]

for table in tables:
    bronze_count = spark.read.format("delta").load(
        f"{bronze_base}/{table}"
    ).count()

    silver_count = spark.read.format("delta").load(
        f"{silver_base}/{table}"
    ).count()

    print(
        f"{table}: Bronze = {bronze_count}, Silver = {silver_count}"
    )

customers: Bronze = 200, Silver = 200
drivers: Bronze = 150, Silver = 150
trucks: Bronze = 120, Silver = 120
trailers: Bronze = 180, Silver = 180
facilities: Bronze = 50, Silver = 50
routes: Bronze = 58, Silver = 58
loads: Bronze = 85410, Silver = 85410
trips: Bronze = 85410, Silver = 85410
fuel_purchases: Bronze = 196442, Silver = 196442
maintenance_records: Bronze = 2920, Silver = 2920
delivery_events: Bronze = 170820, Silver = 170820
safety_incidents: Bronze = 170, Silver = 170
driver_monthly_metrics: Bronze = 4464, Silver = 4464
truck_utilization_metrics: Bronze = 3312, Silver = 3312
